输入构造

首先，我们需要将学生的历史答题序列转化为模型可接受的张量形式。每个交互记录包含：(题目ID，知识点集合，作答正确与否)。我们将其编码为输入张量和目标张量：

输入张量 X 的形状为 [batch_size, seq_len, embed_dim]。其中seq_len是序列长度（每个学生的一系列答题步骤），embed_dim是嵌入维度。我们通过嵌入层将每个交互转换为大小为embed_dim的向量。如果一道题涉及多个知识点，我们会获取每个相关知识点的“知识点+结果”索引，通过嵌入层得到多个向量并将它们相加，得到该交互的合成向量。

目标张量 Y 用于监督训练。对于序列中的每一步，我们需要预测学生对下一题的表现。因此通常采用滑动窗口方式：利用前t步的输入预测第t+1步的结果。在实现中，我们会将序列右移一位作为训练标签。Y表示为与X错开一位的矩阵。对于当前题涉及的知识点，如果答对则这些维度的目标值为1，答错则为0，其余知识点为-1（在计算损失时忽略）。

In [ ]:
config={
    "batch_size": 32,
    "learning_rate": 0.001,
    "num_epochs": 50,
    "num_kc": 100,
}

In [ ]:
# Auto-generated by simulate_students.py
# Students: 100, questions per student: 30

all_student_questions = [[1472,
  213,
  668,
  732,
  176,
  2197,
  39,
  2302,
  128,
  2160,
  387,
  2638,
  2386,
  1909,
  667,
  2410,
  2038,
  680,
  2483,
  1634,
  2639,
  1619,
  945,
  2678,
  848,
  1828,
  191,
  2521,
  2755,
  793],
 [80,
  252,
  2328,
  2024,
  1298,
  522,
  2332,
  792,
  1285,
  1623,
  336,
  633,
  641,
  300,
  1005,
  1257,
  2514,
  2698,
  1629,
  1664,
  214,
  337,
  2761,
  1837,
  1183,
  649,
  927,
  57,
  916,
  620],
 [1956,
  95,
  1814,
  2348,
  417,
  2098,
  2162,
  585,
  1756,
  92,
  1484,
  2771,
  739,
  469,
  1946,
  732,
  1690,
  2481,
  2675,
  1672,
  355,
  1603,
  1647,
  1793,
  684,
  93,
  1672,
  2264,
  2616,
  541],
 [1611,
  2442,
  1664,
  2337,
  886,
  946,
  2551,
  147,
  866,
  1525,
  1965,
  973,
  680,
  721,
  2538,
  680,
  274,
  2283,
  721,
  333,
  1815,
  2405,
  2179,
  2179,
  2741,
  2119,
  2538,
  2335,
  2179,
  2744],
 [502,
  2381,
  1161,
  1261,
  1575,
  29,
  2274,
  1952,
  617,
  1135,
  1391,
  1449,
  2772,
  1845,
  2324,
  363,
  1879,
  790,
  2562,
  812,
  840,
  1316,
  2476,
  501,
  1468,
  2542,
  542,
  2338,
  2351,
  2663],
 [1339,
  118,
  1784,
  257,
  1659,
  1161,
  403,
  2605,
  281,
  2339,
  1375,
  2693,
  461,
  1436,
  1389,
  2082,
  2585,
  1400,
  2681,
  2434,
  2734,
  2193,
  1020,
  2690,
  1449,
  1827,
  1773,
  821,
  1341,
  2394],
 [948,
  705,
  204,
  2150,
  2705,
  2668,
  2014,
  1600,
  160,
  1245,
  163,
  2053,
  708,
  1047,
  23,
  1257,
  858,
  79,
  362,
  710,
  2495,
  2000,
  375,
  1857,
  559,
  404,
  2569,
  2648,
  1860,
  202],
 [864,
  2754,
  2624,
  2464,
  363,
  2706,
  1543,
  2596,
  367,
  1687,
  446,
  1437,
  1864,
  2528,
  1964,
  1757,
  149,
  970,
  1768,
  2258,
  203,
  1723,
  359,
  931,
  2211,
  1516,
  742,
  363,
  148,
  2178],
 [2722,
  343,
  442,
  496,
  2023,
  2285,
  2031,
  1427,
  2057,
  1799,
  2283,
  1477,
  2727,
  1633,
  2702,
  2343,
  1496,
  1375,
  2007,
  2288,
  2245,
  1960,
  1469,
  969,
  755,
  2455,
  2228,
  2374,
  1656,
  2234],
 [2067,
  2678,
  1832,
  441,
  1011,
  993,
  256,
  1808,
  2687,
  843,
  246,
  1380,
  1861,
  1341,
  1429,
  2488,
  2701,
  2511,
  1429,
  2437,
  2004,
  2109,
  259,
  880,
  2701,
  988,
  2701,
  2701,
  2103,
  853],
 [2488,
  2641,
  2507,
  845,
  1747,
  2507,
  1489,
  98,
  1447,
  1304,
  2346,
  563,
  1148,
  2480,
  2385,
  48,
  1762,
  2379,
  2025,
  1783,
  1471,
  1239,
  298,
  2301,
  95,
  1347,
  56,
  2572,
  1275,
  935],
 [80,
  144,
  680,
  323,
  18,
  813,
  625,
  2764,
  942,
  2620,
  1287,
  1710,
  2334,
  1791,
  2735,
  117,
  229,
  2363,
  2416,
  661,
  143,
  744,
  355,
  1516,
  2433,
  753,
  2213,
  1530,
  2689,
  591],
 [1290,
  324,
  588,
  1047,
  779,
  401,
  1325,
  1468,
  2242,
  1983,
  2508,
  1472,
  796,
  2051,
  497,
  159,
  1392,
  769,
  1514,
  2365,
  2028,
  1622,
  2109,
  2172,
  2369,
  2709,
  462,
  1451,
  2049,
  279],
 [514,
  2643,
  2280,
  1658,
  603,
  2325,
  571,
  2459,
  477,
  191,
  2341,
  2678,
  2506,
  1464,
  1622,
  1822,
  2483,
  2406,
  2046,
  2323,
  1427,
  514,
  1015,
  2427,
  1923,
  1864,
  2329,
  2121,
  2140,
  581],
 [1996,
  2632,
  1837,
  288,
  917,
  866,
  2540,
  1686,
  1661,
  2195,
  1055,
  1457,
  2000,
  2568,
  1618,
  1239,
  2642,
  1706,
  2373,
  2678,
  2623,
  1029,
  1534,
  1594,
  1172,
  1618,
  1935,
  2625,
  1819,
  1514],
 [849,
  2427,
  2709,
  646,
  1122,
  1652,
  2653,
  1387,
  2298,
  894,
  2053,
  408,
  2240,
  870,
  1581,
  268,
  397,
  411,
  940,
  2282,
  2613,
  1573,
  2308,
  980,
  2478,
  2440,
  1858,
  1952,
  594,
  160],
 [2367,
  939,
  2642,
  393,
  2175,
  1729,
  2774,
  2367,
  2269,
  2096,
  825,
  1696,
  1972,
  364,
  719,
  2468,
  331,
  830,
  1419,
  2671,
  2562,
  2568,
  1506,
  1463,
  1813,
  2164,
  2625,
  394,
  1833,
  2096],
 [1702,
  229,
  814,
  1719,
  37,
  936,
  2437,
  1031,
  877,
  877,
  298,
  2152,
  551,
  2732,
  229,
  32,
  483,
  49,
  2726,
  1587,
  482,
  872,
  250,
  780,
  2306,
  1428,
  1182,
  2260,
  2028,
  2598],
 [826,
  1478,
  376,
  1037,
  1307,
  317,
  368,
  1411,
  9,
  24,
  2705,
  495,
  288,
  293,
  651,
  339,
  792,
  252,
  1669,
  6,
  339,
  2237,
  305,
  347,
  380,
  2292,
  2117,
  307,
  421,
  255],
 [1341,
  2648,
  1532,
  388,
  180,
  1562,
  259,
  1562,
  2173,
  2464,
  1016,
  2756,
  386,
  156,
  330,
  1808,
  121,
  2778,
  2435,
  381,
  128,
  1535,
  1003,
  1742,
  311,
  193,
  404,
  292,
  109,
  309],
 [2317,
  1938,
  874,
  944,
  2406,
  2174,
  1018,
  995,
  2010,
  374,
  2730,
  2351,
  334,
  1934,
  2368,
  1975,
  1595,
  874,
  2080,
  2236,
  221,
  2659,
  232,
  734,
  2292,
  7,
  2779,
  200,
  2591,
  2488],
 [2681,
  2721,
  2624,
  2267,
  2721,
  2775,
  1842,
  2752,
  2401,
  2438,
  2491,
  2444,
  2350,
  2721,
  2061,
  2477,
  2538,
  2353,
  1818,
  2283,
  959,
  1818,
  1640,
  2551,
  2086,
  956,
  985,
  2002,
  2660,
  1818],
 [2350,
  2725,
  1629,
  2652,
  2366,
  2472,
  23,
  2664,
  2,
  1261,
  1257,
  2443,
  2048,
  2278,
  2705,
  316,
  2472,
  2532,
  2538,
  2382,
  3,
  2149,
  935,
  2657,
  1995,
  1539,
  2547,
  371,
  2485,
  2366],
 [1814,
  2227,
  2659,
  2221,
  2263,
  1316,
  2460,
  2692,
  2650,
  2387,
  2551,
  1180,
  456,
  2734,
  655,
  2551,
  2366,
  2309,
  2766,
  890,
  455,
  1731,
  1975,
  2623,
  2738,
  2623,
  128,
  2061,
  2533,
  1988],
 [919,
  688,
  729,
  2304,
  1698,
  1467,
  425,
  1271,
  1799,
  2412,
  1566,
  648,
  1280,
  2505,
  740,
  924,
  1808,
  2211,
  1868,
  959,
  1430,
  870,
  1551,
  1283,
  985,
  256,
  2601,
  259,
  1038,
  2505],
 [317,
  2487,
  2083,
  1716,
  2611,
  2240,
  2421,
  2164,
  2421,
  2373,
  753,
  21,
  1938,
  144,
  1325,
  1385,
  1522,
  2529,
  2279,
  759,
  2765,
  2008,
  1604,
  1901,
  2706,
  1901,
  468,
  1327,
  1325,
  2333],
 [212,
  423,
  1637,
  2427,
  1628,
  487,
  1741,
  1292,
  1830,
  1839,
  199,
  2725,
  2483,
  1799,
  535,
  1472,
  315,
  2410,
  985,
  341,
  356,
  538,
  465,
  2608,
  334,
  1830,
  788,
  1664,
  1805,
  360],
 [1373,
  2350,
  1338,
  2180,
  1777,
  1155,
  2575,
  2615,
  2260,
  2597,
  2762,
  1387,
  2213,
  2350,
  2350,
  1369,
  1405,
  2260,
  1324,
  2505,
  2505,
  1690,
  758,
  1378,
  2744,
  1381,
  2551,
  1374,
  2621,
  2053],
 [389,
  2359,
  2665,
  2260,
  1730,
  1820,
  1672,
  395,
  411,
  2415,
  1785,
  2344,
  2121,
  2030,
  416,
  1030,
  2688,
  2449,
  946,
  2107,
  1603,
  2470,
  498,
  1778,
  142,
  2412,
  1422,
  2661,
  2665,
  2134],
 [2051,
  1932,
  2037,
  217,
  1782,
  2362,
  2779,
  1783,
  1078,
  2387,
  863,
  476,
  1014,
  1373,
  1947,
  124,
  687,
  1194,
  1795,
  1152,
  14,
  158,
  357,
  125,
  1781,
  72,
  1966,
  341,
  56,
  1970],
 [288,
  2390,
  1827,
  293,
  1196,
  2034,
  289,
  446,
  293,
  2358,
  884,
  511,
  488,
  2356,
  2,
  1961,
  1421,
  2299,
  377,
  1505,
  418,
  2052,
  2118,
  102,
  757,
  1873,
  1798,
  877,
  2558,
  2518],
 [1860,
  687,
  437,
  1007,
  1932,
  1757,
  1992,
  2234,
  358,
  296,
  2481,
  358,
  2276,
  1406,
  289,
  2702,
  2659,
  2251,
  2690,
  2298,
  853,
  2595,
  1536,
  2007,
  1021,
  737,
  1136,
  1841,
  2362,
  2751],
 [1602,
  871,
  321,
  232,
  2036,
  1370,
  1734,
  235,
  2385,
  957,
  1835,
  2559,
  343,
  225,
  2187,
  2260,
  8,
  335,
  8,
  634,
  579,
  1908,
  423,
  2260,
  288,
  2618,
  2037,
  2764,
  1489,
  30],
 [2478,
  1427,
  1468,
  595,
  145,
  1284,
  480,
  572,
  909,
  2046,
  1274,
  1497,
  402,
  699,
  183,
  744,
  915,
  1891,
  772,
  1877,
  2305,
  194,
  1611,
  583,
  2280,
  2370,
  828,
  838,
  972,
  1261],
 [758,
  1964,
  2232,
  675,
  1630,
  676,
  2664,
  668,
  2505,
  1350,
  2392,
  1434,
  1971,
  68,
  1437,
  2499,
  1716,
  936,
  2435,
  1372,
  1075,
  723,
  2171,
  475,
  312,
  1376,
  555,
  403,
  2457,
  1647],
 [1792,
  115,
  654,
  2063,
  2521,
  2608,
  375,
  747,
  150,
  1164,
  906,
  1640,
  1726,
  1259,
  2769,
  217,
  2728,
  571,
  485,
  1496,
  1195,
  933,
  78,
  2278,
  1088,
  982,
  2300,
  2297,
  2716,
  86],
 [1439,
  55,
  1338,
  734,
  128,
  82,
  1310,
  2425,
  1064,
  1591,
  1391,
  887,
  1114,
  88,
  1641,
  2586,
  490,
  1312,
  544,
  798,
  1746,
  1856,
  1300,
  2701,
  1461,
  885,
  1252,
  49,
  1880,
  918],
 [522,
  38,
  1298,
  1,
  115,
  1343,
  95,
  1021,
  492,
  23,
  1541,
  21,
  2195,
  95,
  601,
  609,
  208,
  1498,
  1338,
  728,
  885,
  425,
  228,
  528,
  1342,
  1346,
  2361,
  87,
  228,
  1021],
 [2205,
  750,
  982,
  402,
  0,
  8,
  1104,
  522,
  1504,
  2101,
  326,
  2718,
  2477,
  1756,
  2303,
  1489,
  459,
  949,
  545,
  888,
  2217,
  755,
  270,
  901,
  1622,
  236,
  91,
  726,
  478,
  2033],
 [1772,
  968,
  185,
  933,
  1845,
  288,
  306,
  1860,
  402,
  2207,
  2645,
  2093,
  2033,
  2349,
  270,
  112,
  334,
  14,
  2137,
  1088,
  2507,
  311,
  2120,
  1529,
  2480,
  409,
  655,
  283,
  1691,
  1312],
 [234,
  2246,
  1443,
  1083,
  1767,
  2466,
  2644,
  2141,
  165,
  2041,
  218,
  1610,
  617,
  2660,
  2248,
  2538,
  1017,
  1600,
  229,
  2149,
  168,
  1546,
  165,
  234,
  613,
  169,
  1811,
  2405,
  2440,
  1313],
 [1459,
  2189,
  1280,
  1873,
  1482,
  1566,
  725,
  2436,
  2008,
  1427,
  1256,
  1385,
  2265,
  1192,
  592,
  937,
  520,
  1836,
  768,
  1155,
  2540,
  1880,
  1459,
  514,
  418,
  1575,
  2624,
  368,
  533,
  217],
 [138,
  580,
  2015,
  1933,
  1825,
  808,
  971,
  1456,
  2236,
  926,
  831,
  2055,
  1497,
  1133,
  1435,
  2019,
  2195,
  866,
  2719,
  2460,
  1352,
  12,
  833,
  1258,
  897,
  916,
  1168,
  1175,
  89,
  963],
 [540,
  2127,
  1949,
  2433,
  1047,
  1529,
  595,
  2263,
  24,
  1840,
  533,
  2674,
  431,
  1917,
  2682,
  342,
  143,
  1911,
  378,
  2121,
  5,
  1911,
  348,
  1575,
  120,
  1335,
  2158,
  151,
  548,
  2223],
 [1213,
  283,
  2245,
  2198,
  386,
  273,
  2539,
  1805,
  95,
  2637,
  337,
  233,
  812,
  412,
  1631,
  2176,
  207,
  2160,
  1650,
  1521,
  371,
  588,
  2321,
  379,
  818,
  2123,
  386,
  1034,
  2159,
  730],
 [2726,
  75,
  1845,
  1438,
  2013,
  2756,
  1240,
  2663,
  1773,
  439,
  1029,
  2176,
  2692,
  1844,
  1928,
  285,
  2742,
  2505,
  433,
  2348,
  2535,
  1314,
  958,
  1322,
  235,
  1913,
  2505,
  2013,
  256,
  2172],
 [1343,
  2725,
  318,
  129,
  2259,
  141,
  642,
  667,
  1877,
  1547,
  2531,
  214,
  2094,
  2454,
  447,
  1382,
  931,
  2000,
  1341,
  1347,
  573,
  1783,
  2075,
  1330,
  1746,
  317,
  2566,
  1528,
  1853,
  592],
 [1378,
  127,
  1313,
  2655,
  531,
  2318,
  800,
  1537,
  1201,
  695,
  1119,
  1887,
  790,
  432,
  277,
  1203,
  1550,
  2688,
  2694,
  232,
  2649,
  1503,
  278,
  2088,
  1938,
  2647,
  1865,
  2637,
  1629,
  1206],
 [2547,
  2260,
  2035,
  2362,
  360,
  1515,
  2066,
  843,
  604,
  507,
  1433,
  2409,
  2309,
  702,
  1767,
  258,
  2409,
  816,
  2434,
  1040,
  2435,
  1723,
  1335,
  910,
  380,
  2110,
  1558,
  379,
  1316,
  2547],
 [2239,
  1602,
  519,
  2685,
  1689,
  1767,
  998,
  1473,
  2021,
  2080,
  1648,
  1702,
  2613,
  1106,
  2341,
  2204,
  915,
  2685,
  499,
  2648,
  1982,
  2464,
  507,
  1598,
  499,
  2254,
  2440,
  1810,
  2439,
  1368],
 [868,
  1360,
  2505,
  532,
  2341,
  1866,
  347,
  173,
  237,
  1036,
  669,
  1800,
  2006,
  2285,
  2341,
  279,
  2006,
  2445,
  1471,
  1732,
  2239,
  2521,
  742,
  1350,
  250,
  2394,
  2074,
  911,
  2723,
  358],
 [1479,
  188,
  1450,
  471,
  2113,
  736,
  1979,
  227,
  156,
  584,
  1333,
  260,
  915,
  2155,
  2392,
  295,
  2710,
  2497,
  371,
  520,
  396,
  1831,
  1743,
  1945,
  1033,
  411,
  568,
  420,
  874,
  2601],
 [273,
  427,
  350,
  389,
  2338,
  1560,
  1770,
  1824,
  2444,
  2436,
  482,
  2183,
  830,
  1480,
  2543,
  254,
  296,
  834,
  2183,
  402,
  2634,
  2183,
  2603,
  2444,
  2183,
  453,
  1562,
  2682,
  2505,
  296],
 [2717,
  1914,
  618,
  149,
  147,
  299,
  2660,
  1973,
  1392,
  2744,
  382,
  737,
  1960,
  1812,
  2141,
  204,
  1264,
  1491,
  1474,
  2151,
  612,
  2362,
  1883,
  287,
  252,
  2211,
  1926,
  1729,
  2506,
  1129],
 [1754,
  351,
  1999,
  623,
  2707,
  1799,
  1566,
  2306,
  2236,
  1523,
  333,
  1609,
  2090,
  2491,
  1946,
  2306,
  971,
  2111,
  1918,
  1645,
  1893,
  2117,
  806,
  971,
  163,
  2254,
  1176,
  1018,
  332,
  749],
 [840,
  2221,
  2068,
  2024,
  1681,
  2068,
  2472,
  2121,
  2164,
  1967,
  1622,
  2700,
  2082,
  721,
  1429,
  2360,
  961,
  1754,
  2004,
  786,
  2478,
  2468,
  1899,
  2068,
  1392,
  836,
  2164,
  1332,
  958,
  786],
 [9,
  2219,
  610,
  1646,
  2409,
  614,
  2617,
  1104,
  2635,
  2122,
  1733,
  2722,
  1904,
  2282,
  1298,
  359,
  792,
  2145,
  1763,
  1954,
  1007,
  13,
  2185,
  2409,
  2219,
  793,
  1815,
  189,
  243,
  1345],
 [1952,
  1021,
  2292,
  1020,
  1523,
  1895,
  1561,
  921,
  2440,
  2289,
  382,
  2608,
  882,
  2204,
  1508,
  500,
  2550,
  747,
  2001,
  519,
  2389,
  2669,
  2415,
  2394,
  2779,
  2669,
  1531,
  1606,
  2374,
  2391],
 [1682,
  207,
  25,
  961,
  698,
  854,
  594,
  2392,
  534,
  1700,
  557,
  1575,
  2193,
  200,
  1003,
  2390,
  958,
  1562,
  588,
  2562,
  508,
  1887,
  1947,
  2693,
  1108,
  2305,
  533,
  2124,
  1645,
  2242],
 [2382,
  2043,
  2779,
  1817,
  2386,
  1588,
  2222,
  1702,
  1377,
  1984,
  843,
  1369,
  1453,
  1810,
  1984,
  1704,
  2227,
  1087,
  1747,
  2658,
  2205,
  2660,
  1369,
  2248,
  944,
  1725,
  843,
  1311,
  2586,
  2477],
 [2570,
  688,
  1045,
  161,
  1851,
  1172,
  608,
  253,
  692,
  2090,
  2256,
  2557,
  2563,
  1253,
  1162,
  2339,
  2074,
  1060,
  728,
  738,
  1641,
  654,
  2022,
  1711,
  2505,
  1927,
  1751,
  1402,
  1154,
  1275],
 [1331,
  1617,
  2547,
  1517,
  1698,
  1391,
  39,
  1425,
  473,
  2305,
  1820,
  33,
  1187,
  1531,
  1394,
  58,
  1701,
  31,
  2442,
  2431,
  2031,
  1414,
  1449,
  1614,
  53,
  34,
  2150,
  2244,
  2136,
  47],
 [346,
  643,
  340,
  2137,
  2062,
  714,
  2618,
  246,
  28,
  2608,
  1726,
  2409,
  866,
  904,
  1558,
  2741,
  619,
  325,
  710,
  634,
  2514,
  632,
  327,
  2510,
  644,
  2730,
  2003,
  337,
  1929,
  345],
 [768,
  965,
  1200,
  1973,
  926,
  2156,
  1892,
  1676,
  2339,
  2706,
  896,
  327,
  309,
  982,
  982,
  2293,
  1425,
  2773,
  650,
  1129,
  594,
  557,
  1908,
  650,
  577,
  1047,
  2219,
  1657,
  2152,
  1836],
 [2378,
  808,
  2668,
  1694,
  844,
  956,
  2222,
  2053,
  1907,
  2246,
  1683,
  749,
  123,
  826,
  694,
  1304,
  2265,
  1255,
  497,
  975,
  1368,
  1341,
  1344,
  2031,
  1337,
  2575,
  2539,
  1969,
  2343,
  1474],
 [1904,
  2453,
  1372,
  1891,
  2656,
  739,
  249,
  322,
  655,
  2501,
  721,
  2395,
  145,
  870,
  562,
  2351,
  1547,
  434,
  125,
  971,
  2592,
  1821,
  139,
  433,
  473,
  442,
  698,
  1809,
  1826,
  2663],
 [2029,
  1917,
  379,
  2296,
  2029,
  517,
  2434,
  413,
  1428,
  86,
  1920,
  157,
  99,
  965,
  927,
  1426,
  668,
  619,
  630,
  2037,
  448,
  122,
  1879,
  1951,
  2520,
  420,
  618,
  2090,
  796,
  1057],
 [337,
  1446,
  2305,
  984,
  2756,
  972,
  2636,
  2703,
  1574,
  2381,
  2586,
  326,
  425,
  1802,
  1339,
  1448,
  1809,
  1868,
  1341,
  126,
  2528,
  2745,
  1059,
  1040,
  1743,
  1214,
  1369,
  2104,
  2693,
  1911],
 [2639,
  2454,
  1297,
  1345,
  821,
  2387,
  252,
  1385,
  566,
  2121,
  1681,
  294,
  102,
  1282,
  2004,
  372,
  1982,
  180,
  2165,
  425,
  485,
  1392,
  2221,
  2477,
  1369,
  1332,
  2165,
  2592,
  255,
  365],
 [2283,
  479,
  2421,
  2123,
  370,
  1520,
  340,
  2302,
  2323,
  355,
  27,
  2382,
  2409,
  41,
  230,
  2431,
  953,
  2366,
  319,
  116,
  2472,
  2405,
  2632,
  2445,
  40,
  2067,
  2043,
  2726,
  2156,
  2200],
 [1669,
  358,
  534,
  2368,
  487,
  237,
  722,
  1327,
  1057,
  569,
  2391,
  1379,
  657,
  2669,
  2165,
  928,
  2556,
  2753,
  125,
  2246,
  2490,
  532,
  1684,
  2012,
  254,
  1017,
  142,
  1751,
  243,
  2169],
 [2226,
  810,
  1976,
  261,
  693,
  958,
  261,
  1490,
  2336,
  1970,
  1827,
  1976,
  627,
  325,
  54,
  888,
  174,
  972,
  2291,
  293,
  1515,
  1984,
  550,
  1622,
  696,
  78,
  1247,
  2261,
  541,
  239],
 [103,
  2386,
  1965,
  1713,
  889,
  1848,
  1070,
  625,
  1984,
  2443,
  2654,
  1577,
  336,
  207,
  2337,
  91,
  140,
  2607,
  996,
  1041,
  1046,
  1915,
  1210,
  760,
  1431,
  23,
  1193,
  930,
  1541,
  1296],
 [1449,
  1298,
  253,
  2239,
  1557,
  1769,
  2607,
  1442,
  1264,
  362,
  1732,
  664,
  2757,
  264,
  2564,
  925,
  2039,
  2013,
  1021,
  2006,
  2371,
  2286,
  633,
  1400,
  1491,
  375,
  2703,
  686,
  1374,
  2356],
 [1743,
  913,
  173,
  2014,
  262,
  568,
  1026,
  1701,
  2250,
  2237,
  530,
  1817,
  285,
  2222,
  1400,
  2049,
  2477,
  462,
  2030,
  277,
  729,
  211,
  892,
  2698,
  2668,
  2639,
  924,
  2289,
  1548,
  1024],
 [396,
  875,
  2375,
  213,
  2347,
  1997,
  200,
  680,
  1464,
  1811,
  17,
  239,
  866,
  147,
  2660,
  101,
  1642,
  2215,
  460,
  264,
  1024,
  1391,
  96,
  2327,
  2576,
  812,
  1339,
  1856,
  1177,
  1604],
 [2058,
  373,
  265,
  2307,
  960,
  1928,
  2194,
  2563,
  1426,
  2365,
  2763,
  2114,
  2138,
  1478,
  594,
  900,
  843,
  2692,
  2052,
  1273,
  597,
  1449,
  1834,
  912,
  2153,
  1136,
  802,
  1530,
  1631,
  2725],
 [1036,
  741,
  480,
  297,
  277,
  278,
  193,
  1560,
  2741,
  1009,
  571,
  872,
  2718,
  1830,
  472,
  2172,
  790,
  926,
  413,
  1428,
  2152,
  932,
  1048,
  693,
  497,
  1006,
  2343,
  878,
  2174,
  1718],
 [1402,
  39,
  1723,
  2409,
  2635,
  2094,
  1830,
  225,
  2535,
  2229,
  2201,
  2741,
  2036,
  1944,
  1110,
  2217,
  1044,
  1828,
  2661,
  1433,
  1320,
  2471,
  742,
  1489,
  137,
  1280,
  252,
  2762,
  1051,
  2145],
 [738,
  2649,
  894,
  827,
  187,
  952,
  579,
  1213,
  288,
  992,
  904,
  2693,
  924,
  1101,
  2267,
  1760,
  2136,
  951,
  2365,
  1376,
  1768,
  683,
  692,
  1433,
  894,
  641,
  165,
  1964,
  2673,
  1375],
 [1323,
  2299,
  2597,
  307,
  2038,
  323,
  2083,
  1992,
  2449,
  1444,
  321,
  2008,
  354,
  398,
  2236,
  2178,
  1879,
  1624,
  1743,
  1799,
  2364,
  423,
  2008,
  1444,
  2056,
  1726,
  1130,
  440,
  2033,
  386],
 [629,
  1938,
  2656,
  2483,
  2305,
  1668,
  2135,
  125,
  801,
  1590,
  1535,
  454,
  1673,
  919,
  2603,
  608,
  2068,
  2378,
  869,
  1439,
  2141,
  2008,
  1003,
  689,
  325,
  2343,
  298,
  2551,
  1685,
  2621],
 [535,
  734,
  1382,
  2761,
  2265,
  2565,
  143,
  2179,
  1337,
  143,
  1369,
  956,
  1298,
  2024,
  968,
  2727,
  1341,
  1830,
  2599,
  2244,
  1369,
  2394,
  583,
  1379,
  830,
  2472,
  140,
  1911,
  2538,
  826],
 [2068,
  2308,
  411,
  1761,
  2778,
  2258,
  1819,
  995,
  2107,
  191,
  2447,
  2300,
  8,
  2068,
  1221,
  1875,
  365,
  1489,
  847,
  2734,
  1875,
  882,
  1514,
  2177,
  2214,
  4,
  847,
  1489,
  421,
  612],
 [1602,
  518,
  993,
  2053,
  793,
  2413,
  185,
  1803,
  557,
  2368,
  2739,
  2678,
  1323,
  2693,
  599,
  1288,
  2165,
  510,
  2527,
  2591,
  1786,
  1388,
  1934,
  1295,
  2050,
  393,
  2355,
  204,
  958,
  501],
 [1651,
  810,
  980,
  2109,
  1405,
  223,
  333,
  264,
  1521,
  1322,
  1669,
  861,
  1661,
  2503,
  2715,
  372,
  209,
  1207,
  225,
  228,
  800,
  2338,
  1479,
  1490,
  43,
  401,
  1521,
  1292,
  398,
  1531],
 [273,
  2771,
  148,
  952,
  2187,
  1743,
  1568,
  1380,
  1595,
  1648,
  2370,
  2493,
  2696,
  1605,
  2709,
  2026,
  2765,
  1996,
  2036,
  667,
  882,
  1957,
  2561,
  2761,
  280,
  912,
  1648,
  271,
  997,
  1778],
 [731,
  646,
  553,
  631,
  1635,
  2401,
  288,
  98,
  1376,
  350,
  2650,
  1574,
  270,
  336,
  2214,
  362,
  308,
  491,
  468,
  288,
  357,
  1812,
  360,
  1380,
  380,
  631,
  1076,
  2068,
  1558,
  1869],
 [790,
  1875,
  1533,
  1681,
  1379,
  2387,
  1538,
  302,
  2307,
  2376,
  1743,
  1407,
  1783,
  905,
  1297,
  1799,
  1661,
  2172,
  1482,
  1287,
  1758,
  2194,
  1983,
  975,
  1383,
  1895,
  2037,
  1430,
  1375,
  2648],
 [2451,
  1460,
  1170,
  2175,
  1192,
  781,
  351,
  1914,
  1268,
  79,
  2025,
  1836,
  27,
  2478,
  612,
  1985,
  587,
  176,
  2116,
  1552,
  2381,
  2636,
  343,
  2548,
  1785,
  891,
  1981,
  182,
  1582,
  2596],
 [327,
  2397,
  1657,
  612,
  362,
  2529,
  804,
  640,
  792,
  2186,
  2105,
  1896,
  22,
  1602,
  991,
  870,
  116,
  2668,
  168,
  1602,
  1000,
  187,
  1692,
  1478,
  1634,
  325,
  2121,
  2241,
  1795,
  2617],
 [2714,
  1344,
  858,
  2356,
  2435,
  1340,
  2606,
  594,
  1881,
  1522,
  930,
  1946,
  350,
  1349,
  2156,
  2770,
  2107,
  1376,
  1369,
  2601,
  456,
  1654,
  1298,
  1346,
  1339,
  457,
  375,
  1369,
  435,
  1329],
 [2071,
  1839,
  1692,
  879,
  535,
  698,
  1787,
  1873,
  2298,
  574,
  1708,
  762,
  2424,
  235,
  298,
  1889,
  235,
  2601,
  2230,
  168,
  1856,
  2756,
  589,
  980,
  2200,
  1298,
  956,
  477,
  2461,
  262],
 [724,
  2466,
  1333,
  2489,
  523,
  1574,
  450,
  2139,
  2434,
  438,
  2710,
  2751,
  1634,
  2284,
  224,
  26,
  572,
  260,
  1713,
  1144,
  30,
  958,
  14,
  533,
  203,
  931,
  1944,
  2520,
  1120,
  1678],
 [2342,
  22,
  2713,
  2052,
  1221,
  1727,
  1877,
  827,
  994,
  1485,
  1488,
  2689,
  2684,
  2547,
  694,
  2319,
  1768,
  2249,
  2452,
  2078,
  634,
  2058,
  2198,
  2072,
  1726,
  2668,
  3,
  1880,
  2602,
  1464],
 [607,
  1400,
  2722,
  296,
  2660,
  2424,
  2182,
  1664,
  998,
  2163,
  1702,
  1763,
  1582,
  2180,
  1960,
  2156,
  1182,
  2043,
  1261,
  617,
  1857,
  1730,
  2457,
  609,
  125,
  2633,
  553,
  2647,
  2484,
  790],
 [316,
  2246,
  1323,
  2289,
  2543,
  1041,
  905,
  1911,
  1330,
  1323,
  1612,
  363,
  1222,
  362,
  2031,
  1648,
  1003,
  749,
  1453,
  1552,
  1349,
  1038,
  812,
  506,
  1505,
  1330,
  144,
  469,
  916,
  5],
 [371,
  2550,
  1549,
  1133,
  399,
  614,
  380,
  460,
  2239,
  2239,
  2407,
  2355,
  2326,
  250,
  15,
  14,
  1644,
  1674,
  213,
  460,
  806,
  1854,
  1485,
  855,
  1877,
  615,
  530,
  248,
  663,
  1274],
 [1910,
  2413,
  1008,
  2333,
  29,
  162,
  104,
  2576,
  184,
  1967,
  707,
  384,
  192,
  2348,
  997,
  2606,
  24,
  2223,
  385,
  2564,
  2147,
  215,
  2128,
  1994,
  383,
  2379,
  398,
  26,
  540,
  370],
 [2614,
  580,
  76,
  995,
  2650,
  2017,
  302,
  93,
  1505,
  2583,
  349,
  580,
  965,
  2114,
  1035,
  139,
  1469,
  414,
  109,
  473,
  1935,
  1152,
  2725,
  287,
  2640,
  1995,
  611,
  593,
  320,
  2505]]

all_student_answers = [[1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1],
 [1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0],
 [1,
  1,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  1],
 [0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1],
 [0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  0],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  1],
 [0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1],
 [1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0],
 [0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1],
 [0,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0],
 [0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0],
 [0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  0],
 [1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1],
 [1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1],
 [0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  0],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0],
 [1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [0,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1],
 [1,
  0,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1],
 [1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [1,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0],
 [1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1],
 [1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  1,
  0,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0],
 [1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  0,
  0,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1],
 [1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 [1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1],
 [1,
  0,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1],
 [1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1],
 [0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1],
 [0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1],
 [1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  0,
  1]]

Q_matrix = {0: [75],
 1: [75],
 2: [81, 82],
 3: [81, 110, 75],
 4: [81, 75],
 5: [110, 75],
 6: [75],
 7: [75],
 8: [82, 81],
 9: [75],
 10: [75, 110, 59],
 11: [75, 110, 40],
 12: [75],
 13: [110, 67, 75],
 14: [82, 11],
 15: [75],
 16: [75],
 17: [82, 110, 40],
 18: [75, 20, 25, 40],
 19: [110, 67, 75],
 20: [110, 67, 75],
 21: [110, 75],
 22: [82, 100, 40],
 23: [40, 75],
 24: [75, 40, 69, 110],
 25: [75],
 26: [82, 40, 110],
 27: [110, 75],
 28: [75, 58],
 29: [82, 69, 40, 75, 25, 20],
 30: [82, 24, 15, 20],
 31: [42],
 32: [42],
 33: [42],
 34: [42, 45],
 35: [42],
 36: [42, 117, 78],
 37: [42, 117],
 38: [42, 110],
 39: [42, 117, 110],
 40: [42, 110],
 41: [42, 110],
 42: [42, 110],
 43: [42],
 44: [42, 78],
 45: [42, 78],
 46: [42, 78, 20],
 47: [42, 110],
 48: [42, 110, 78],
 49: [42, 110, 78],
 50: [42],
 51: [42, 78, 110],
 52: [42, 110],
 53: [42, 110],
 54: [42, 110],
 55: [42, 110, 78],
 56: [42, 117, 110],
 57: [42, 117, 110],
 58: [40, 42, 20, 0],
 59: [40, 42, 20],
 60: [36],
 61: [36],
 62: [36],
 63: [100],
 64: [100],
 65: [100, 48],
 66: [100],
 67: [100],
 68: [40, 74],
 69: [52, 73],
 70: [100, 78],
 71: [100],
 72: [100],
 73: [100, 40],
 74: [100, 74, 78],
 75: [100, 73],
 76: [100, 74],
 77: [100],
 78: [100],
 79: [100, 74, 20, 40],
 80: [73, 100, 25, 20, 110],
 81: [25, 100, 73],
 82: [25, 100, 73],
 83: [25, 100, 73],
 84: [100],
 85: [47, 100],
 86: [100],
 87: [100, 42, 20],
 88: [100, 42, 0, 20],
 89: [100, 20, 73],
 90: [100, 74, 11, 20],
 91: [100, 74, 20, 78],
 92: [100, 20, 78],
 93: [100, 20, 78],
 94: [100, 59, 20, 67],
 95: [100, 59, 20],
 96: [100, 47, 48],
 97: [100, 20, 73],
 98: [100, 47, 20, 36],
 99: [100, 20],
 100: [100, 47, 59, 20],
 101: [46, 100, 47, 20],
 102: [100, 47, 20, 59],
 103: [100, 47, 40],
 104: [100, 47, 20],
 105: [100, 47, 74],
 106: [100, 47],
 107: [100, 47, 74],
 108: [100, 47, 20],
 109: [100, 47, 20],
 110: [100, 47, 20],
 111: [100, 47, 20],
 112: [100, 47, 20],
 113: [100, 47, 20],
 114: [100, 47, 59, 20],
 115: [100, 47, 59, 20],
 116: [100, 47, 20, 58],
 117: [100, 47, 20, 67],
 118: [100, 40, 113, 47, 20],
 119: [40, 47, 100],
 120: [100, 40, 47, 20, 18],
 121: [100, 47, 20, 58],
 122: [100, 47, 20, 58],
 123: [100, 119],
 124: [100, 74, 78],
 125: [100, 74, 78],
 126: [20, 119, 73],
 127: [100, 74, 20],
 128: [119, 74, 106, 14],
 129: [119, 120],
 130: [65],
 131: [65],
 132: [60, 61],
 133: [60],
 134: [128],
 135: [65],
 136: [65],
 137: [65],
 138: [61, 78, 20],
 139: [60, 78, 20],
 140: [60, 78, 20],
 141: [78, 20],
 142: [78, 20, 45],
 143: [60, 61, 78, 20],
 144: [61, 78, 20, 110],
 145: [78, 110],
 146: [60, 104, 78],
 147: [61, 78],
 148: [61, 78, 20],
 149: [61, 78, 20],
 150: [61, 78, 20, 110],
 151: [61, 78, 20],
 152: [60, 61, 117],
 153: [61, 20, 78],
 154: [48, 110, 78, 91],
 155: [74, 73],
 156: [74, 20, 48],
 157: [74, 47, 20],
 158: [74, 47, 40, 20],
 159: [47, 74, 40],
 160: [74, 59, 20, 67],
 161: [40, 78, 110, 65],
 162: [40, 59],
 163: [48, 40, 20, 91],
 164: [39, 40, 65],
 165: [39, 40, 65],
 166: [39, 58, 48],
 167: [40],
 168: [39, 40],
 169: [39, 40],
 170: [40, 78],
 171: [40, 73],
 172: [40, 0, 20, 78],
 173: [40, 0],
 174: [52, 0, 78],
 175: [74, 40],
 176: [40, 74, 78],
 177: [40, 74],
 178: [40, 74],
 179: [40, 73],
 180: [40, 74],
 181: [40, 11, 78],
 182: [40, 0, 78, 20],
 183: [40, 73, 0],
 184: [40, 73, 20],
 185: [25, 40, 73, 11],
 186: [40, 25],
 187: [40, 25],
 188: [40, 73, 20],
 189: [40, 47, 73, 20],
 190: [40, 11, 73, 20],
 191: [73, 40, 11, 20],
 192: [40, 73, 20, 0],
 193: [74, 40, 20, 11, 18],
 194: [40, 11, 18, 73],
 195: [46, 47, 74],
 196: [40, 47, 73],
 197: [40, 47, 73],
 198: [40, 47, 0],
 199: [40, 47, 11],
 200: [40, 47, 11, 110],
 201: [40, 11, 18, 47, 20],
 202: [40, 11, 18, 78],
 203: [40, 113, 47, 11],
 204: [40, 113, 47, 11],
 205: [74, 47, 20, 40],
 206: [40, 47, 11],
 207: [40, 47, 74, 20, 11],
 208: [40, 47, 74, 0],
 209: [40, 47, 73, 11, 20],
 210: [40, 113, 47, 11],
 211: [40, 113, 47, 11],
 212: [40, 47, 11],
 213: [40, 47, 11, 73],
 214: [40, 47, 69, 20],
 215: [40, 47, 74, 2, 20],
 216: [40, 47, 11, 73],
 217: [58, 69, 47, 20],
 218: [52, 53],
 219: [52, 53],
 220: [52, 53],
 221: [52, 53],
 222: [52, 53],
 223: [53, 20, 0],
 224: [52, 53, 69, 11, 20],
 225: [52, 53, 78, 0],
 226: [53, 113, 11, 20],
 227: [52, 11, 18, 113],
 228: [53, 0, 20],
 229: [52, 53, 58, 59, 11, 20, 18],
 230: [52, 74, 58],
 231: [74, 39, 52],
 232: [74, 52, 53],
 233: [52, 74],
 234: [52, 74],
 235: [52, 73],
 236: [52, 73],
 237: [52, 25, 20, 0],
 238: [48],
 239: [48],
 240: [48, 46, 106],
 241: [48],
 242: [46],
 243: [124, 27, 48],
 244: [48, 46, 47, 65],
 245: [48],
 246: [48, 20, 106, 18, 115],
 247: [47, 18, 11, 48, 78],
 248: [48, 46, 47, 20],
 249: [46, 47, 123, 20, 78],
 250: [124, 11, 47],
 251: [123, 47, 11, 78],
 252: [47, 28, 11],
 253: [47, 124, 104],
 254: [124, 47],
 255: [46, 47, 28],
 256: [123, 47, 40, 20],
 257: [47, 11, 78, 48],
 258: [46, 27, 11, 20],
 259: [123, 11, 20, 104],
 260: [47, 73, 11, 20, 28, 78],
 261: [48],
 262: [48, 47, 20, 110],
 263: [48, 47, 40, 11, 20],
 264: [47, 20, 48],
 265: [46, 47, 11],
 266: [48, 18, 11, 47, 20],
 267: [74, 46],
 268: [46, 74],
 269: [47, 74],
 270: [115, 47],
 271: [115, 47, 110, 78],
 272: [115, 20, 18],
 273: [115, 20, 47],
 274: [115, 47],
 275: [115, 20, 18],
 276: [115, 20, 18, 78],
 277: [115, 110, 18],
 278: [115, 110, 18],
 279: [47, 115, 20, 18],
 280: [115, 47, 18, 78],
 281: [115, 47, 18, 78],
 282: [115, 40, 47, 20],
 283: [115, 47, 18],
 284: [115, 40, 18, 47],
 285: [115, 40, 20, 18],
 286: [115, 47, 74],
 287: [115, 40, 47, 20, 18],
 288: [58, 59, 110, 25, 20],
 289: [58, 59, 56, 79],
 290: [91, 58],
 291: [58, 59],
 292: [58, 59],
 293: [58, 59, 48, 106],
 294: [58, 59],
 295: [58, 74],
 296: [58, 20, 32],
 297: [58, 11, 18, 59],
 298: [58, 11, 18, 117],
 299: [58, 20, 40],
 300: [58, 20, 40],
 301: [58, 11, 18, 20],
 302: [58, 11, 18, 20],
 303: [58, 74, 106],
 304: [74, 58],
 305: [58, 59, 74],
 306: [48, 58, 74],
 307: [58, 59, 74],
 308: [58, 74],
 309: [74, 58, 25, 20],
 310: [58, 74],
 311: [58, 74],
 312: [58, 20, 78],
 313: [58, 73, 20],
 314: [58, 69, 20, 78],
 315: [58, 74, 110],
 316: [20, 58, 47, 110],
 317: [20, 59, 47, 110],
 318: [58, 47, 74, 20, 69],
 319: [110, 58, 69, 73, 78],
 320: [58, 20, 11, 18, 73],
 321: [110, 25],
 322: [110, 58, 25, 38],
 323: [58, 74, 106],
 324: [110, 58, 25, 17, 69],
 325: [58, 25, 110],
 326: [110, 58, 25, 17, 20],
 327: [110, 25, 20],
 328: [58, 47, 18, 38, 110, 78],
 329: [18, 11, 110],
 330: [58, 110, 69, 78],
 331: [52, 58, 74],
 332: [58, 73],
 333: [58, 74, 73],
 334: [58, 74, 53],
 335: [52, 59, 20],
 336: [58, 110, 11, 25, 20],
 337: [59, 110, 53, 20, 25],
 338: [59, 110, 54, 11, 38],
 339: [52, 59, 69, 38, 110],
 340: [58, 20, 11],
 341: [58, 20, 73, 0],
 342: [58, 20, 11],
 343: [52, 58, 11, 20, 78],
 344: [58, 47, 74],
 345: [58, 59, 110, 20, 25],
 346: [58, 59, 67, 33],
 347: [18, 54, 30, 55],
 348: [18, 58, 67],
 349: [18, 54, 67, 20],
 350: [58, 47, 67],
 351: [58, 59, 69, 73],
 352: [58, 20, 67],
 353: [20, 54, 58],
 354: [20, 67, 58, 54],
 355: [58, 20, 67, 40],
 356: [58, 20, 67, 78],
 357: [58, 20, 67, 78],
 358: [58, 20, 32, 26],
 359: [58, 20, 33, 26],
 360: [58, 20, 32, 26],
 361: [20, 74, 78],
 362: [58, 55, 20, 31, 29],
 363: [58, 59, 110, 55, 69, 2],
 364: [54, 33, 67, 20, 73],
 365: [58, 59, 20, 67, 73],
 366: [55, 33, 31],
 367: [40, 59, 11, 67, 20],
 368: [59, 73, 69, 2],
 369: [58, 46, 74, 38, 104],
 370: [58, 69, 104, 73, 46],
 371: [58, 46, 74, 104, 69, 2],
 372: [59, 47, 73, 104],
 373: [59, 47, 74],
 374: [58, 46, 89, 69, 73],
 375: [54, 31, 11, 18, 55],
 376: [113, 59, 11, 18, 67],
 377: [58, 55, 20, 67],
 378: [58, 18, 37, 67],
 379: [59, 20, 67, 32],
 380: [58, 59, 55, 67, 32],
 381: [74, 47, 65],
 382: [74, 73],
 383: [74, 73],
 384: [74, 73],
 385: [74, 73],
 386: [74, 20, 78],
 387: [74, 73, 106],
 388: [74, 40],
 389: [47, 74],
 390: [74, 106],
 391: [74],
 392: [74],
 393: [25, 74],
 394: [25, 74, 110],
 395: [48, 74, 25, 20, 110],
 396: [48, 78, 74, 20],
 397: [74, 78, 20],
 398: [74, 73],
 399: [46, 74, 78],
 400: [74, 104],
 401: [40, 73, 104, 20],
 402: [74, 104, 20, 78],
 403: [48, 74, 20, 78],
 404: [40, 74, 20],
 405: [40, 74, 104, 20],
 406: [47, 74],
 407: [74, 104, 47, 20],
 408: [74, 104, 47, 20],
 409: [47, 74],
 410: [47, 74],
 411: [47, 74, 14],
 412: [47, 74],
 413: [47, 74, 18, 20, 48],
 414: [47, 74, 20],
 415: [74, 47, 20, 78],
 416: [47, 20, 74],
 417: [47, 74, 20],
 418: [58, 47, 74, 104, 20, 54],
 419: [73, 59, 47, 38, 54],
 420: [48, 47, 20, 74, 104],
 421: [59, 47, 54],
 422: [58, 47, 74],
 423: [128, 67, 20, 25],
 424: [20, 65],
 425: [78, 67, 9, 7],
 426: [47, 46, 65],
 427: [47],
 428: [74, 47, 65],
 429: [47, 46],
 430: [46, 47, 67],
 431: [46, 67],
 432: [46, 74, 100],
 433: [106, 46, 47],
 434: [46, 47, 23],
 435: [46, 47],
 436: [47],
 437: [47],
 438: [46, 40],
 439: [40, 46, 47],
 440: [47, 67],
 441: [47, 58, 91],
 442: [47, 0, 110, 78],
 443: [20, 11, 47],
 444: [40, 47, 38, 20],
 445: [11, 47, 20, 78],
 446: [18, 11, 104, 47],
 447: [11, 46, 20, 78],
 448: [47, 18, 78, 20],
 449: [47, 18, 78],
 450: [47, 78, 20, 46],
 451: [47, 78],
 452: [47, 78, 104, 20],
 453: [47, 78],
 454: [47, 78, 20, 91],
 455: [46, 40],
 456: [46, 47],
 457: [46, 47],
 458: [47, 46],
 459: [47, 104],
 460: [40, 47, 11],
 461: [54, 33, 89],
 462: [54, 89, 26],
 463: [48, 58],
 464: [58, 54],
 465: [54, 58],
 466: [58, 54],
 467: [58, 54],
 468: [58, 54],
 469: [54, 110, 109],
 470: [109],
 471: [58, 54],
 472: [117, 89, 67, 78],
 473: [78, 58, 54],
 474: [20, 48, 78],
 475: [58, 78, 110],
 476: [78, 20, 58],
 477: [18, 117, 110],
 478: [69, 20, 58, 78],
 479: [58, 38, 110, 78],
 480: [54, 47, 18, 38, 58, 17, 22, 55],
 481: [18, 11, 20, 110],
 482: [58, 20, 18, 37],
 483: [54, 58],
 484: [48],
 485: [58, 59],
 486: [110, 78, 58],
 487: [58, 18, 37, 20, 78],
 488: [78, 20, 59],
 489: [37, 58, 20, 78],
 490: [20, 21, 58, 56, 109],
 491: [110, 11, 104, 58],
 492: [11, 110, 59],
 493: [11, 20, 110],
 494: [11, 18, 110, 58],
 495: [11, 18, 110, 59],
 496: [38, 117, 78, 89],
 497: [38, 110, 117, 89],
 498: [38, 117],
 499: [38],
 500: [38, 110],
 501: [69, 2],
 502: [69, 104, 38],
 503: [38, 104, 78],
 504: [38, 104],
 505: [58, 38, 18, 78, 20, 54],
 506: [48, 69, 78, 110, 38],
 507: [48, 104, 38, 110],
 508: [104, 117, 2],
 509: [69, 2],
 510: [69, 2, 104, 78],
 511: [69, 38, 78, 104],
 512: [110, 69, 38, 104],
 513: [69, 104, 38],
 514: [5, 104, 20, 117],
 515: [69, 104, 38, 78],
 516: [69, 2, 104, 78],
 517: [58, 37, 20, 104],
 518: [110, 69, 2],
 519: [18, 69, 38],
 520: [69, 38, 104, 48],
 521: [69, 2, 0],
 522: [69, 0],
 523: [11, 69],
 524: [69, 0],
 525: [69, 0],
 526: [11],
 527: [38, 11],
 528: [69, 0],
 529: [20, 11, 106],
 530: [69, 11, 48, 110, 20],
 531: [11, 20, 110],
 532: [0, 78],
 533: [69, 11, 104, 78],
 534: [40, 11, 78, 20],
 535: [60, 11, 18, 40, 46],
 536: [69, 110, 11, 38],
 537: [11, 18, 78, 110],
 538: [11, 18, 104],
 539: [11, 18, 104],
 540: [11, 104, 18],
 541: [48, 69, 78, 20],
 542: [48, 69, 11, 104, 78, 110, 20],
 543: [11, 20, 110],
 544: [11, 18, 78, 110],
 545: [11, 18, 104],
 546: [11, 18, 104],
 547: [52, 53, 11, 78],
 548: [11, 104],
 549: [11, 78, 110],
 550: [11, 20, 69],
 551: [11, 20, 110],
 552: [3, 4],
 553: [35, 69, 4, 3],
 554: [110, 11, 18],
 555: [110, 18, 0],
 556: [11, 18, 20, 111],
 557: [11, 3, 4, 111],
 558: [117],
 559: [117, 78],
 560: [117, 78, 89],
 561: [117, 104],
 562: [42, 117, 78],
 563: [117, 78],
 564: [78, 89, 117],
 565: [117, 48, 78],
 566: [117, 48, 78],
 567: [117, 78, 48],
 568: [48, 78, 117, 110],
 569: [104, 38, 117, 78],
 570: [38, 104, 117, 78, 48],
 571: [117, 78, 44],
 572: [18, 117, 104, 89],
 573: [18, 1],
 574: [18, 117, 89],
 575: [18, 106, 89],
 576: [18, 117, 106, 89],
 577: [18, 78],
 578: [110, 18, 78, 89],
 579: [25, 20, 18, 78, 110],
 580: [18, 117, 78, 89],
 581: [18, 117, 110],
 582: [18, 110, 51],
 583: [18, 78, 20],
 584: [48, 18, 78, 110],
 585: [18, 11, 20],
 586: [18, 110, 89],
 587: [18, 104, 110],
 588: [18, 117, 78, 110],
 589: [18, 117, 78, 110],
 590: [18, 110, 89],
 591: [18, 117, 48, 110],
 592: [18, 106, 78, 20, 48],
 593: [48, 18, 78, 110],
 594: [18, 78, 67, 9],
 595: [117, 104, 18],
 596: [89, 117, 104],
 597: [18, 117, 78],
 598: [18, 117, 89],
 599: [18, 117, 78],
 600: [117, 104],
 601: [117, 104, 18],
 602: [58, 117, 78, 20],
 603: [11, 20, 117],
 604: [47, 78, 20, 18],
 605: [104, 117, 106, 20],
 606: [35, 106],
 607: [35],
 608: [35, 78, 91],
 609: [35, 11, 59, 20, 110],
 610: [110, 35, 38],
 611: [35, 11, 110, 104],
 612: [35, 11, 69],
 613: [35, 37, 110, 25, 20],
 614: [35, 69, 11],
 615: [35, 11, 40],
 616: [35, 106, 78],
 617: [35, 69, 38],
 618: [35, 5, 58, 20],
 619: [110, 20, 25],
 620: [110, 25, 20],
 621: [20, 58, 56],
 622: [20, 58, 78, 110],
 623: [20, 91, 110],
 624: [20, 110],
 625: [20, 78, 110],
 626: [20, 110],
 627: [48, 78, 110, 104],
 628: [20, 110],
 629: [20, 109],
 630: [110, 25, 20, 58, 109],
 631: [25, 58, 110],
 632: [25, 110, 48],
 633: [110, 25],
 634: [25, 110],
 635: [25, 110],
 636: [110, 25],
 637: [25, 110, 91],
 638: [110, 25, 74],
 639: [25, 74, 110],
 640: [25, 58, 110],
 641: [110, 40, 73, 25],
 642: [25, 20, 110],
 643: [25, 110, 48],
 644: [110, 25],
 645: [25, 110],
 646: [25, 110, 91],
 647: [110, 25, 20, 58],
 648: [25, 91, 110],
 649: [110, 25],
 650: [25, 100, 73],
 651: [18, 106, 78],
 652: [92],
 653: [46, 47, 92],
 654: [46, 92, 65],
 655: [46, 92],
 656: [92],
 657: [91, 78, 65],
 658: [47, 46, 67],
 659: [46, 47],
 660: [46, 47],
 661: [106],
 662: [65],
 663: [106],
 664: [106, 90],
 665: [106],
 666: [106, 92],
 667: [128, 20, 106],
 668: [106, 20, 78],
 669: [48],
 670: [65],
 671: [85],
 672: [65],
 673: [65],
 674: [65],
 675: [85],
 676: [85],
 677: [65],
 678: [65],
 679: [48],
 680: [61, 66],
 681: [56, 65],
 682: [56, 109],
 683: [56, 109],
 684: [56, 109],
 685: [91, 56, 109],
 686: [56],
 687: [91, 56],
 688: [91, 78],
 689: [48, 91, 106],
 690: [48, 91, 106],
 691: [48],
 692: [91],
 693: [48, 25, 20],
 694: [91],
 695: [65],
 696: [48, 104, 20, 78],
 697: [48, 104, 20, 110],
 698: [48, 110, 78],
 699: [48, 78, 110, 20],
 700: [48, 78, 20],
 701: [48, 65],
 702: [26, 20, 91],
 703: [91],
 704: [91],
 705: [56],
 706: [91],
 707: [91],
 708: [56, 91],
 709: [91],
 710: [91, 56],
 711: [91, 78],
 712: [78, 91],
 713: [117, 104, 78],
 714: [45],
 715: [45],
 716: [78],
 717: [78],
 718: [78, 91],
 719: [78],
 720: [78],
 721: [48, 78, 122],
 722: [78, 48],
 723: [78],
 724: [78],
 725: [78, 104, 89],
 726: [78, 104],
 727: [78],
 728: [78, 67, 91],
 729: [78, 89, 48, 91],
 730: [78, 104],
 731: [78, 58, 59, 54, 55, 78],
 732: [78, 104],
 733: [78, 89, 117],
 734: [78, 109, 56],
 735: [78, 20, 91],
 736: [48, 78, 106, 110, 111],
 737: [106, 74],
 738: [20, 78, 91, 111],
 739: [78, 20, 111],
 740: [91, 110, 20, 111],
 741: [78, 110, 111],
 742: [106, 78, 26],
 743: [106, 78],
 744: [78, 117, 89],
 745: [106, 78, 89, 104],
 746: [106, 78],
 747: [78, 106, 108],
 748: [106, 78],
 749: [108, 98, 104, 44],
 750: [78, 104, 91],
 751: [106, 78, 89],
 752: [106, 104, 78],
 753: [78, 117, 45],
 754: [78],
 755: [78, 104],
 756: [78, 91, 89],
 757: [78, 20, 104, 111],
 758: [104, 78],
 759: [104, 111],
 760: [104, 20, 111],
 761: [48, 78, 104, 111],
 762: [18, 104, 111],
 763: [18, 104, 111],
 764: [38, 104, 111],
 765: [38, 104, 111],
 766: [104, 111],
 767: [104, 111],
 768: [104, 111],
 769: [11, 18, 104, 111],
 770: [104, 18, 111],
 771: [18, 104, 111],
 772: [117, 91, 100],
 773: [65],
 774: [65],
 775: [92],
 776: [65],
 777: [92],
 778: [65],
 779: [92],
 780: [92],
 781: [78, 65],
 782: [65],
 783: [65],
 784: [110, 65],
 785: [110, 97],
 786: [106, 18, 35, 69, 38, 11, 3, 4],
 787: [65],
 788: [106, 18, 35, 69, 11, 38],
 789: [46, 47],
 790: [106, 18, 69, 35],
 791: [92],
 792: [106, 18, 35, 69, 46, 100, 65],
 793: [106, 18, 35, 69, 11, 38, 92],
 794: [92],
 795: [65],
 796: [106, 18, 35, 69, 2, 0],
 797: [65],
 798: [92],
 799: [65],
 800: [78, 67, 91],
 801: [65],
 802: [47, 78, 111],
 803: [18, 35, 69],
 804: [25, 110],
 805: [46, 74],
 806: [106, 18, 35, 69, 38, 11],
 807: [18, 117, 111],
 808: [65],
 809: [65],
 810: [52, 53, 79],
 811: [47, 78, 111],
 812: [69, 40, 113],
 813: [92],
 814: [65],
 815: [47, 78, 111],
 816: [26],
 817: [40, 73, 20, 111],
 818: [74],
 819: [100],
 820: [65],
 821: [65],
 822: [18, 78, 117],
 823: [78, 110, 111],
 824: [48, 91, 74],
 825: [62, 92],
 826: [119, 100],
 827: [128, 110, 20, 25],
 828: [20, 91, 110, 111],
 829: [52, 79],
 830: [119, 100],
 831: [74, 47, 65],
 832: [65],
 833: [65],
 834: [69, 18, 78, 110, 47],
 835: [48],
 836: [20, 111],
 837: [100],
 838: [106, 18, 69],
 839: [74, 73],
 840: [102, 48, 106, 110, 78],
 841: [82, 75],
 842: [110, 25, 20],
 843: [100, 20, 79, 111],
 844: [78, 45],
 845: [100, 79],
 846: [82, 81],
 847: [29, 55],
 848: [119],
 849: [39],
 850: [82],
 851: [3],
 852: [42],
 853: [58, 56, 79],
 854: [117],
 855: [46],
 856: [78, 91],
 857: [91],
 858: [56],
 859: [47, 74],
 860: [65],
 861: [73, 100],
 862: [65],
 863: [42],
 864: [128, 61],
 865: [47, 11, 110],
 866: [128, 20, 60, 61, 111],
 867: [35, 111],
 868: [78, 104, 108],
 869: [56, 109],
 870: [25, 91, 110],
 871: [25, 110],
 872: [11, 104],
 873: [46, 47],
 874: [53],
 875: [40, 47],
 876: [55, 20, 33, 31],
 877: [54, 30, 11, 104, 117, 89],
 878: [58, 11, 18, 111],
 879: [18, 108, 111],
 880: [18, 1, 51],
 881: [11, 47, 111],
 882: [11, 38, 117, 78],
 883: [78, 108, 104],
 884: [42, 110, 111],
 885: [42, 78],
 886: [78, 108],
 887: [48, 91],
 888: [56, 109, 33],
 889: [47, 40, 48],
 890: [44, 78],
 891: [117, 44, 78],
 892: [40, 73, 20, 111],
 893: [35, 11, 117, 110, 78],
 894: [35, 91, 110, 78],
 895: [110, 20, 78, 111],
 896: [20, 111],
 897: [47, 115, 111],
 898: [52, 11, 20, 79],
 899: [35, 69, 5, 111],
 900: [52, 40, 0, 18, 73, 25, 20],
 901: [85],
 902: [85],
 903: [20, 111],
 904: [25, 110],
 905: [58, 110, 44, 55],
 906: [92],
 907: [0, 117, 104],
 908: [47],
 909: [56, 109, 0, 54],
 910: [25, 74],
 911: [47, 18, 117, 0],
 912: [18, 58, 38, 54, 17, 117],
 913: [110, 58, 11, 21],
 914: [78, 117, 111],
 915: [18, 38, 78, 110, 58],
 916: [110, 78, 111],
 917: [92],
 918: [110, 25, 20],
 919: [78, 91, 48],
 920: [108, 78],
 921: [100, 25, 108],
 922: [69, 37, 108, 78],
 923: [78, 108, 104],
 924: [110, 25, 58, 54],
 925: [56, 109, 33],
 926: [20, 111],
 927: [110, 20, 111],
 928: [110, 78, 18, 111],
 929: [18, 51, 110, 78],
 930: [48, 69, 78, 110, 67, 38],
 931: [11, 20, 111],
 932: [53, 11, 20, 111],
 933: [40, 73, 11, 61, 25, 20],
 934: [45],
 935: [117, 44],
 936: [78, 108],
 937: [38, 104, 111],
 938: [11, 20, 111],
 939: [18, 108, 44],
 940: [74],
 941: [47, 74, 40],
 942: [60, 61, 111],
 943: [100],
 944: [111, 22, 58],
 945: [52, 73],
 946: [54, 31, 74, 47, 55],
 947: [82],
 948: [74, 40, 108],
 949: [66],
 950: [78],
 951: [26, 48],
 952: [78, 48, 44],
 953: [3],
 954: [82],
 955: [27],
 956: [74, 119],
 957: [74, 52],
 958: [11, 4, 3, 18, 111],
 959: [47, 123, 111],
 960: [115, 20, 111],
 961: [53, 11, 18, 20, 111],
 962: [69, 18, 78, 110],
 963: [110, 59, 11, 55, 33],
 964: [82, 20, 37],
 965: [58, 37, 18, 111],
 966: [18, 117, 110, 78],
 967: [47, 73, 0, 78],
 968: [66],
 969: [78],
 970: [26, 48],
 971: [78, 91, 108],
 972: [3],
 973: [75, 82],
 974: [27],
 975: [74, 119],
 976: [74, 52],
 977: [11, 4, 3, 111],
 978: [123, 47, 20, 11, 111],
 979: [115, 20, 111],
 980: [53, 11, 18, 111],
 981: [69, 18, 78, 110],
 982: [58, 11, 25, 20, 55],
 983: [82, 110, 37],
 984: [36, 35, 18],
 985: [78, 67, 9],
 986: [46, 92],
 987: [65],
 988: [52, 79],
 989: [20, 111],
 990: [31, 55, 37, 104, 106],
 991: [27],
 992: [38, 5, 117, 44],
 993: [3, 4],
 994: [78, 42, 110, 111],
 995: [18, 38, 59, 117],
 996: [52, 11, 78, 111],
 997: [40, 47, 61, 20, 111],
 998: [69, 38, 108, 104, 97],
 999: [100, 74],
 1000: [25, 20],
 1001: [48],
 1002: [78, 108, 104],
 1003: [78, 98, 91],
 1004: [42, 18, 111],
 1005: [69, 38, 5],
 1006: [60, 61],
 1007: [56, 109, 33],
 1008: [73, 25, 11],
 1009: [78, 110, 111],
 1010: [20, 111],
 1011: [115, 116, 78, 110, 67],
 1012: [11, 18, 117, 78],
 1013: [11, 48, 78, 111],
 1014: [58, 54, 37, 111],
 1015: [18, 117, 110, 78],
 1016: [46, 47, 74, 11],
 1017: [36, 18, 35],
 1018: [78, 91, 108],
 1019: [20, 111],
 1020: [48, 108, 58, 109],
 1021: [58, 59, 56, 20, 111],
 1022: [90],
 1023: [11, 104],
 1024: [40, 47],
 1025: [20, 74, 58, 111],
 1026: [74, 52],
 1027: [47, 18, 78, 111],
 1028: [58, 55, 22, 111],
 1029: [59, 11, 18, 54, 22, 17],
 1030: [107, 18, 108, 111],
 1031: [42],
 1032: [106, 18, 69],
 1033: [48],
 1034: [78],
 1035: [78, 98],
 1036: [54, 30, 111],
 1037: [11, 18, 110, 78],
 1038: [25, 110, 91],
 1039: [25, 110],
 1040: [52, 26],
 1041: [78, 110, 111],
 1042: [20, 111],
 1043: [18, 110, 117, 78],
 1044: [110, 78, 1, 111],
 1045: [110, 18, 69, 37, 111],
 1046: [123, 47, 11, 20],
 1047: [110, 40, 2, 111],
 1048: [47, 40, 61, 111],
 1049: [65],
 1050: [65],
 1051: [65],
 1052: [65],
 1053: [65],
 1054: [65],
 1055: [65],
 1056: [65],
 1057: [65],
 1058: [65],
 1059: [65],
 1060: [65],
 1061: [65],
 1062: [65],
 1063: [65],
 1064: [65],
 1065: [65],
 1066: [65],
 1067: [65],
 1068: [65],
 1069: [65],
 1070: [65],
 1071: [65],
 1072: [65],
 1073: [65],
 1074: [65],
 1075: [65],
 1076: [65],
 1077: [100],
 1078: [100],
 1079: [65],
 1080: [65],
 1081: [65],
 1082: [65],
 1083: [65],
 1084: [65],
 1085: [65],
 1086: [65],
 1087: [65],
 1088: [65],
 1089: [65],
 1090: [65],
 1091: [65],
 1092: [65],
 1093: [65],
 1094: [65],
 1095: [65],
 1096: [65],
 1097: [65],
 1098: [65],
 1099: [65],
 1100: [44],
 1101: [44],
 1102: [44],
 1103: [65],
 1104: [65],
 1105: [65],
 1106: [44],
 1107: [65],
 1108: [44, 117],
 1109: [65],
 1110: [65],
 1111: [65],
 1112: [65],
 1113: [65],
 1114: [65],
 1115: [65],
 1116: [65],
 1117: [65],
 1118: [65],
 1119: [65],
 1120: [65],
 1121: [65],
 1122: [65],
 1123: [65],
 1124: [65],
 1125: [65],
 1126: [65],
 1127: [65],
 1128: [65],
 1129: [65],
 1130: [65],
 1131: [65],
 1132: [65],
 1133: [65],
 1134: [65],
 1135: [65],
 1136: [65],
 1137: [65],
 1138: [65],
 1139: [65],
 1140: [65],
 1141: [65],
 1142: [65],
 1143: [65],
 1144: [65],
 1145: [65],
 1146: [65],
 1147: [65],
 1148: [100],
 1149: [65],
 1150: [65],
 1151: [65],
 1152: [65],
 1153: [65],
 1154: [65],
 1155: [65],
 1156: [65],
 1157: [65],
 1158: [65],
 1159: [65],
 1160: [65],
 1161: [65],
 1162: [65],
 1163: [65],
 1164: [65],
 1165: [65],
 1166: [65],
 1167: [65],
 1168: [65],
 1169: [65],
 1170: [65],
 1171: [65],
 1172: [65],
 1173: [92],
 1174: [92],
 1175: [65],
 1176: [65],
 1177: [65],
 1178: [65],
 1179: [65],
 1180: [65],
 1181: [92],
 1182: [92],
 1183: [65],
 1184: [92],
 1185: [65],
 1186: [65],
 1187: [92],
 1188: [65],
 1189: [65],
 1190: [65],
 1191: [62],
 1192: [62],
 1193: [65],
 1194: [65],
 1195: [92],
 1196: [93],
 1197: [65],
 1198: [65],
 1199: [65],
 1200: [65],
 1201: [65],
 1202: [65],
 1203: [92, 65],
 1204: [65],
 1205: [65],
 1206: [65],
 1207: [65],
 1208: [65],
 1209: [92],
 1210: [65],
 1211: [65],
 1212: [65],
 1213: [65],
 1214: [65],
 1215: [65],
 1216: [65],
 1217: [92],
 1218: [65],
 1219: [65],
 1220: [65],
 1221: [65],
 1222: [92],
 1223: [65],
 1224: [92],
 1225: [92],
 1226: [65],
 1227: [92],
 1228: [65],
 1229: [65],
 1230: [65],
 1231: [65],
 1232: [62],
 1233: [65],
 1234: [65],
 1235: [65],
 1236: [65],
 1237: [65],
 1238: [65],
 1239: [92],
 1240: [65],
 1241: [65],
 1242: [65],
 1243: [65],
 1244: [65],
 1245: [65],
 1246: [65],
 1247: [65],
 1248: [65],
 1249: [93],
 1250: [92],
 1251: [65],
 1252: [92, 97],
 1253: [65],
 1254: [65],
 1255: [65],
 1256: [62],
 1257: [90],
 1258: [92, 90],
 1259: [92],
 1260: [65],
 1261: [18, 35, 69],
 1262: [66, 48],
 1263: [80],
 1264: [58, 56, 109],
 1265: [62],
 1266: [62],
 1267: [62],
 1268: [62],
 1269: [62],
 1270: [62],
 1271: [62],
 1272: [62],
 1273: [62],
 1274: [106, 18, 35, 69],
 1275: [62, 92],
 1276: [92],
 1277: [106],
 1278: [106, 18, 35, 69, 2, 0],
 1279: [62],
 1280: [62],
 1281: [65],
 1282: [97],
 1283: [65],
 1284: [97],
 1285: [65],
 1286: [65],
 1287: [97],
 1288: [97],
 1289: [65],
 1290: [65],
 1291: [65],
 1292: [97],
 1293: [65],
 1294: [97],
 1295: [97],
 1296: [65],
 1297: [119, 120],
 1298: [119, 120],
 1299: [119, 100],
 1300: [119, 100],
 1301: [119, 100],
 1302: [76],
 1303: [76],
 1304: [119, 120],
 1305: [65],
 1306: [39, 40],
 1307: [65],
 1308: [65],
 1309: [65],
 1310: [65],
 1311: [65],
 1312: [119, 100],
 1313: [39, 40, 78, 117, 65],
 1314: [115, 47],
 1315: [86],
 1316: [71],
 1317: [65],
 1318: [119],
 1319: [86, 73],
 1320: [74, 87],
 1321: [86],
 1322: [69, 40, 73],
 1323: [86, 70, 84],
 1324: [86],
 1325: [70, 99],
 1326: [70],
 1327: [65],
 1328: [97],
 1329: [86, 48, 78],
 1330: [70],
 1331: [70, 69, 78],
 1332: [40, 69, 97],
 1333: [70, 113],
 1334: [70, 40, 78],
 1335: [86, 71],
 1336: [86, 121],
 1337: [119, 120],
 1338: [119, 120],
 1339: [119],
 1340: [119],
 1341: [119, 120],
 1342: [119, 120],
 1343: [119, 120],
 1344: [119, 99, 76],
 1345: [119, 120],
 1346: [119, 120],
 1347: [119, 120],
 1348: [119, 74],
 1349: [119, 70],
 1350: [70, 125, 126, 118],
 1351: [119, 120],
 1352: [65],
 1353: [65],
 1354: [65],
 1355: [65],
 1356: [119, 116],
 1357: [65],
 1358: [65],
 1359: [65],
 1360: [65],
 1361: [65],
 1362: [65],
 1363: [65],
 1364: [65],
 1365: [65],
 1366: [65],
 1367: [65],
 1368: [65],
 1369: [116, 119, 120],
 1370: [116],
 1371: [116],
 1372: [116],
 1373: [116],
 1374: [116],
 1375: [116, 119, 115],
 1376: [116, 119, 100],
 1377: [116],
 1378: [115, 116],
 1379: [115, 116, 119],
 1380: [115, 116, 111],
 1381: [115, 116],
 1382: [119, 120, 97],
 1383: [97],
 1384: [40, 119, 97],
 1385: [41, 99],
 1386: [41],
 1387: [41],
 1388: [97],
 1389: [113],
 1390: [113],
 1391: [50, 40, 77],
 1392: [72, 40, 97],
 1393: [113],
 1394: [113],
 1395: [97],
 1396: [97],
 1397: [86, 47],
 1398: [86],
 1399: [86],
 1400: [86, 119],
 1401: [86],
 1402: [65],
 1403: [65],
 1404: [65],
 1405: [116, 86],
 1406: [50, 77],
 1407: [97],
 1408: [92],
 1409: [62],
 1410: [86],
 1411: [106, 18, 35, 69],
 1412: [65],
 1413: [65],
 1414: [65],
 1415: [65],
 1416: [65],
 1417: [65],
 1418: [85],
 1419: [92],
 1420: [117, 104, 105],
 1421: [85, 107],
 1422: [85],
 1423: [78, 105, 80],
 1424: [25, 110],
 1425: [25, 110, 80],
 1426: [35, 11, 20, 110, 111],
 1427: [18, 105, 44, 66, 106],
 1428: [11, 104, 111],
 1429: [11, 3, 4, 111],
 1430: [38, 44, 110, 107],
 1431: [40, 79, 66],
 1432: [48, 40, 58],
 1433: [26],
 1434: [40, 59, 79, 107],
 1435: [74, 47, 111],
 1436: [47, 23, 85],
 1437: [47, 78, 111],
 1438: [25, 17, 38, 11, 59, 110],
 1439: [74, 98],
 1440: [47, 74, 111],
 1441: [74, 47, 23, 111],
 1442: [58, 74, 56, 79],
 1443: [128],
 1444: [67, 128, 46],
 1445: [82, 11, 111],
 1446: [36],
 1447: [100, 107],
 1448: [100, 46, 30, 11, 23],
 1449: [54, 55, 37, 18, 73, 47, 23, 107],
 1450: [97],
 1451: [86, 70],
 1452: [65],
 1453: [99, 86, 70],
 1454: [62],
 1455: [65],
 1456: [65],
 1457: [65],
 1458: [65],
 1459: [44],
 1460: [65],
 1461: [92],
 1462: [109],
 1463: [108, 105, 104],
 1464: [44, 105, 78],
 1465: [78, 98, 108],
 1466: [42],
 1467: [20, 91, 111],
 1468: [18, 117, 111],
 1469: [35, 11, 110, 111],
 1470: [11, 104, 111],
 1471: [11, 20, 111],
 1472: [42, 40, 11, 20, 111],
 1473: [52, 79],
 1474: [74, 98],
 1475: [46, 74, 107],
 1476: [40, 61, 107],
 1477: [52, 53, 11, 78, 111],
 1478: [25, 17, 38, 11, 59, 110, 54],
 1479: [60, 53, 47, 111],
 1480: [47, 20, 111],
 1481: [58, 47, 55, 111],
 1482: [74, 106, 66],
 1483: [74, 108],
 1484: [74],
 1485: [67, 128, 46],
 1486: [128, 61],
 1487: [100, 107],
 1488: [100, 47, 111],
 1489: [82, 81],
 1490: [52, 53, 47, 11],
 1491: [46, 11, 113, 59, 55, 110, 107],
 1492: [65],
 1493: [65],
 1494: [65],
 1495: [65],
 1496: [18, 35, 69],
 1497: [92],
 1498: [65],
 1499: [65],
 1500: [65],
 1501: [65],
 1502: [106, 97],
 1503: [65],
 1504: [78, 65],
 1505: [108, 107],
 1506: [104, 105, 111],
 1507: [106, 108, 105],
 1508: [20, 58, 56, 111],
 1509: [42, 108],
 1510: [18, 117, 106],
 1511: [18, 117, 111],
 1512: [11, 21, 111],
 1513: [11, 18, 104, 111],
 1514: [97, 92],
 1515: [52, 53, 79],
 1516: [117, 44, 37, 105],
 1517: [52, 79, 107],
 1518: [40, 61, 107],
 1519: [47, 85, 23],
 1520: [58, 59, 11, 18, 54, 111],
 1521: [54, 30, 56, 89],
 1522: [40, 47, 124, 111],
 1523: [40, 73, 20, 111],
 1524: [74, 47, 111],
 1525: [82],
 1526: [74, 44, 107],
 1527: [67, 128],
 1528: [128],
 1529: [100],
 1530: [100, 47, 111],
 1531: [42, 111],
 1532: [69, 78, 37, 23],
 1533: [46, 69, 11, 91, 23, 98, 107],
 1534: [72, 77],
 1535: [76, 99],
 1536: [65],
 1537: [65],
 1538: [93, 99, 76, 97],
 1539: [106, 90],
 1540: [65],
 1541: [65],
 1542: [65],
 1543: [65],
 1544: [46, 67, 47],
 1545: [65],
 1546: [85, 66],
 1547: [20, 111],
 1548: [66, 56, 109],
 1549: [104, 108, 105],
 1550: [20, 78, 111],
 1551: [20, 111, 91],
 1552: [44, 105, 104],
 1553: [35, 11, 110, 111],
 1554: [11, 104, 111],
 1555: [38, 5],
 1556: [42, 40, 111],
 1557: [30, 55, 18],
 1558: [56, 58, 79, 55],
 1559: [128, 20, 25, 67],
 1560: [47, 74, 111],
 1561: [47, 105, 111],
 1562: [47, 23, 85],
 1563: [60, 53, 11, 18, 111],
 1564: [25, 40, 37],
 1565: [74, 106, 20],
 1566: [74, 44, 107],
 1567: [74, 108, 65],
 1568: [74, 47, 23, 65],
 1569: [128],
 1570: [60, 20, 105, 111],
 1571: [100, 107],
 1572: [100, 47, 20, 111],
 1573: [54, 33, 11],
 1574: [108, 11, 91, 105, 23],
 1575: [48, 117, 37, 18, 23, 104, 108, 107, 111],
 1576: [106, 18, 35, 69],
 1577: [65],
 1578: [65],
 1579: [65],
 1580: [106],
 1581: [76, 41, 77],
 1582: [86],
 1583: [65],
 1584: [65],
 1585: [108, 107, 44, 78],
 1586: [92],
 1587: [46, 92, 97],
 1588: [25, 110, 78, 80],
 1589: [85],
 1590: [109],
 1591: [78, 67, 66, 98, 105],
 1592: [17, 25, 38, 54, 58],
 1593: [110, 21, 20, 25, 111],
 1594: [35, 11, 110, 111],
 1595: [18, 105, 117, 44],
 1596: [11, 111],
 1597: [11, 104, 111],
 1598: [42, 40, 11, 111],
 1599: [52, 79],
 1600: [48, 39, 58],
 1601: [30, 18, 55, 89],
 1602: [25, 17, 38, 11, 54, 59],
 1603: [47, 23, 85],
 1604: [47, 40, 111],
 1605: [115, 20, 111],
 1606: [74, 108, 65],
 1607: [47, 74],
 1608: [74, 47, 78, 65],
 1609: [74, 98, 107],
 1610: [128],
 1611: [60, 61, 111, 78],
 1612: [100, 107],
 1613: [100, 47, 111],
 1614: [42, 111],
 1615: [20, 111],
 1616: [11, 78, 23],
 1617: [11, 18, 54, 55, 47, 107],
 1618: [86, 76],
 1619: [65],
 1620: [65],
 1621: [65],
 1622: [92, 93],
 1623: [65],
 1624: [65],
 1625: [65],
 1626: [65],
 1627: [65],
 1628: [46, 67, 47],
 1629: [90],
 1630: [85],
 1631: [110, 20, 111],
 1632: [78, 98, 111],
 1633: [105, 78],
 1634: [25, 110],
 1635: [20, 91, 111],
 1636: [46, 19],
 1637: [35, 11, 110, 111],
 1638: [11, 111],
 1639: [11, 104, 111],
 1640: [93, 97, 65],
 1641: [92, 97, 93],
 1642: [40, 73, 21, 111],
 1643: [46, 47],
 1644: [74, 46],
 1645: [46, 47, 23],
 1646: [47, 78, 111],
 1647: [47, 23, 85],
 1648: [25, 17, 38, 11, 59, 110],
 1649: [58, 74, 54, 33],
 1650: [54, 58, 74, 108, 56],
 1651: [74, 53, 47, 111],
 1652: [74, 98],
 1653: [128, 61, 65],
 1654: [67, 128, 46],
 1655: [100, 107],
 1656: [100, 47, 31, 111],
 1657: [100, 47, 111],
 1658: [80, 117, 37, 74, 47],
 1659: [107, 18, 69, 104, 23],
 1660: [125],
 1661: [97, 77],
 1662: [86, 70],
 1663: [62],
 1664: [18, 35, 69],
 1665: [65],
 1666: [65],
 1667: [65],
 1668: [99, 76],
 1669: [65],
 1670: [18, 35, 69, 46, 100],
 1671: [65],
 1672: [85],
 1673: [105, 78],
 1674: [48, 91, 108, 107],
 1675: [106, 108],
 1676: [25, 110],
 1677: [25, 110, 91],
 1678: [11, 110, 111],
 1679: [58, 20, 117, 110, 105, 111],
 1680: [11, 104, 111],
 1681: [97, 92],
 1682: [46, 47, 62],
 1683: [11, 111],
 1684: [30, 55, 18, 89],
 1685: [74, 66],
 1686: [128, 20, 25, 67],
 1687: [47, 23, 85],
 1688: [46, 74, 40],
 1689: [47, 18, 78, 111],
 1690: [39, 40, 60, 79],
 1691: [74, 107],
 1692: [74, 47, 66, 25, 20],
 1693: [40, 74, 105, 107],
 1694: [74, 98],
 1695: [128],
 1696: [60, 20, 105, 111],
 1697: [36],
 1698: [100, 107],
 1699: [100, 36],
 1700: [37, 18, 78, 108, 23],
 1701: [108, 18, 37, 47, 110, 80],
 1702: [86, 70, 84, 99, 125],
 1703: [76, 125],
 1704: [70, 76],
 1705: [62],
 1706: [62],
 1707: [65],
 1708: [92, 65],
 1709: [65],
 1710: [65],
 1711: [65],
 1712: [92],
 1713: [46, 47, 65],
 1714: [20, 111],
 1715: [18, 111, 66, 44],
 1716: [85],
 1717: [85, 107],
 1718: [20, 111],
 1719: [20, 111],
 1720: [74, 98, 48, 91],
 1721: [11, 111],
 1722: [101],
 1723: [26, 58, 78],
 1724: [11, 111],
 1725: [40, 79, 91, 78],
 1726: [128, 20, 25, 67],
 1727: [54, 33, 59, 56, 79],
 1728: [30, 18, 55],
 1729: [25, 40, 37],
 1730: [47, 85, 23],
 1731: [46, 74, 40, 47],
 1732: [27, 28, 124],
 1733: [74, 106, 107],
 1734: [82, 74],
 1735: [74, 73, 106, 66],
 1736: [47, 74, 14, 111],
 1737: [67, 46, 128],
 1738: [128],
 1739: [100, 107],
 1740: [65],
 1741: [82, 67, 110],
 1742: [80, 37, 18, 78, 23],
 1743: [61, 53, 20, 11, 23],
 1744: [70, 99],
 1745: [50],
 1746: [70, 77],
 1747: [66, 80],
 1748: [86],
 1749: [65],
 1750: [65],
 1751: [65],
 1752: [65],
 1753: [76, 77],
 1754: [101, 93, 97],
 1755: [65],
 1756: [85],
 1757: [78, 108, 105],
 1758: [117, 44, 78],
 1759: [109],
 1760: [20, 91, 111],
 1761: [25, 110, 90],
 1762: [18, 117, 111],
 1763: [35, 11, 110, 111],
 1764: [11, 104, 111],
 1765: [69, 11, 38, 106],
 1766: [38, 104, 111],
 1767: [52, 79],
 1768: [74, 26, 111],
 1769: [30, 18, 55],
 1770: [46, 47, 46],
 1771: [47, 20, 105, 112],
 1772: [47, 46, 65],
 1773: [47, 78, 23, 111],
 1774: [40, 47, 11, 111],
 1775: [74, 104, 108],
 1776: [74, 47, 111],
 1777: [74, 47, 78, 65],
 1778: [74, 66],
 1779: [128],
 1780: [67, 46],
 1781: [100, 47, 20, 111],
 1782: [100, 107],
 1783: [100, 47, 20, 111],
 1784: [69, 18, 78, 110, 23],
 1785: [47, 69, 74, 78, 107],
 1786: [97],
 1787: [93],
 1788: [76, 77],
 1789: [62, 92],
 1790: [86],
 1791: [65],
 1792: [92],
 1793: [85],
 1794: [76, 77],
 1795: [100, 76, 77],
 1796: [46, 67, 47, 65],
 1797: [92],
 1798: [85, 107],
 1799: [78, 67, 91, 105, 66],
 1800: [108, 106, 105, 107],
 1801: [20, 111, 66],
 1802: [20, 91, 111],
 1803: [25, 110],
 1804: [44, 117, 105],
 1805: [58, 110, 78, 117, 111],
 1806: [65],
 1807: [11, 111],
 1808: [48, 27, 123],
 1809: [36],
 1810: [39, 40, 79],
 1811: [48, 39, 40, 58, 59],
 1812: [56, 52, 58, 74, 107],
 1813: [47, 105, 111],
 1814: [47, 23, 85],
 1815: [82, 75],
 1816: [47, 78, 20, 111],
 1817: [106, 74, 94, 66],
 1818: [94, 74],
 1819: [74, 73],
 1820: [74, 52, 47, 111],
 1821: [67, 46, 128],
 1822: [128],
 1823: [100, 107],
 1824: [47, 100, 36, 111],
 1825: [42, 110, 111],
 1826: [48, 37, 18, 20, 25, 23, 78, 117, 110],
 1827: [48, 58, 11, 33, 55, 47, 25, 20, 107],
 1828: [100, 74, 26],
 1829: [0, 78, 108],
 1830: [58, 33, 26, 111],
 1831: [100, 36, 26],
 1832: [39, 40, 79],
 1833: [100, 25, 20, 73],
 1834: [65],
 1835: [115, 20, 23, 111],
 1836: [5, 104, 111],
 1837: [92],
 1838: [100, 47, 20, 31, 111],
 1839: [47, 67],
 1840: [40, 0, 78],
 1841: [60, 105, 111],
 1842: [59, 21, 105, 111],
 1843: [65],
 1844: [48, 105, 104, 111],
 1845: [46, 23],
 1846: [35, 11, 111],
 1847: [52, 53, 20, 111],
 1848: [78, 80, 105],
 1849: [65],
 1850: [65],
 1851: [78, 91],
 1852: [52, 47, 20, 111],
 1853: [119, 120],
 1854: [11, 4, 111],
 1855: [100, 65],
 1856: [40, 47, 73],
 1857: [127, 114, 68],
 1858: [66],
 1859: [115, 86, 47],
 1860: [56, 109, 66],
 1861: [125, 76],
 1862: [20, 111],
 1863: [65],
 1864: [108, 117, 104, 105],
 1865: [25, 58, 110],
 1866: [108, 107],
 1867: [65],
 1868: [48, 11, 91, 78, 23],
 1869: [11, 91, 20, 23],
 1870: [52, 53, 47, 20, 111],
 1871: [42, 110, 78],
 1872: [86],
 1873: [100, 73, 104, 20, 111],
 1874: [41],
 1875: [119, 14],
 1876: [18, 111, 108],
 1877: [52, 11, 18, 20, 47, 111],
 1878: [5, 18, 3, 111],
 1879: [74, 47, 23, 111],
 1880: [18, 104, 111],
 1881: [40, 74, 107],
 1882: [47, 40, 111],
 1883: [60, 61, 128],
 1884: [65],
 1885: [40, 11, 47, 111],
 1886: [65],
 1887: [115, 23, 18, 80, 111],
 1888: [92],
 1889: [46, 47, 23],
 1890: [58, 33],
 1891: [117, 78, 48, 105],
 1892: [11, 20, 111],
 1893: [48, 106, 91, 107],
 1894: [65],
 1895: [56, 66, 109],
 1896: [93],
 1897: [110, 25],
 1898: [117, 105, 104],
 1899: [93],
 1900: [65],
 1901: [70, 99, 121],
 1902: [65],
 1903: [47],
 1904: [78, 105],
 1905: [106],
 1906: [40, 105, 104],
 1907: [100, 108, 111],
 1908: [82, 24, 111],
 1909: [99, 125, 76],
 1910: [82, 110, 69, 54, 47],
 1911: [119, 47, 37, 23, 74, 100, 48, 69],
 1912: [42, 111],
 1913: [123, 27, 124],
 1914: [40, 74],
 1915: [100, 104, 20, 111],
 1916: [18, 20, 11, 111],
 1917: [48, 47, 11, 20, 111],
 1918: [58, 20, 18, 37, 111],
 1919: [82, 111],
 1920: [47, 78, 111],
 1921: [60, 20, 105, 111],
 1922: [110, 25, 20, 111],
 1923: [60, 117, 44],
 1924: [86, 47],
 1925: [65],
 1926: [74, 47, 111],
 1927: [65],
 1928: [17, 25, 38, 54, 58],
 1929: [25, 74, 110, 91],
 1930: [58, 56, 74, 108],
 1931: [74, 47, 111],
 1932: [66, 56],
 1933: [65],
 1934: [76, 125],
 1935: [18, 78, 105],
 1936: [50, 77],
 1937: [65],
 1938: [86, 121, 70, 84, 99, 125, 126],
 1939: [11, 111],
 1940: [18, 110, 111],
 1941: [48],
 1942: [38, 11, 110, 105, 107],
 1943: [65],
 1944: [18, 117, 104, 111],
 1945: [58, 5, 111],
 1946: [85],
 1947: [58, 20, 11, 111],
 1948: [65],
 1949: [20, 111],
 1950: [105, 108, 104],
 1951: [47, 18, 78, 111],
 1952: [69, 40, 58, 18, 38],
 1953: [113, 11, 40, 47],
 1954: [82, 75],
 1955: [74, 111],
 1956: [100, 47, 20, 31, 112],
 1957: [65],
 1958: [47, 104, 111],
 1959: [60, 105, 111],
 1960: [35, 11, 40, 111],
 1961: [42, 111],
 1962: [107, 57],
 1963: [100, 107],
 1964: [58, 33, 26, 20, 111],
 1965: [108, 11, 18, 61, 23],
 1966: [74, 58, 56, 25],
 1967: [93],
 1968: [55, 74, 37],
 1969: [78, 98, 105],
 1970: [58, 54, 20, 33, 111],
 1971: [70, 121, 126],
 1972: [124, 47, 105, 111],
 1973: [35, 11, 111],
 1974: [65],
 1975: [58, 59, 54, 110, 25, 20, 17, 22, 55],
 1976: [74, 52, 53],
 1977: [18, 117, 105, 44],
 1978: [108, 105],
 1979: [58, 11, 18, 111],
 1980: [101],
 1981: [48, 78, 110, 111],
 1982: [97, 77],
 1983: [92, 97, 93, 101],
 1984: [86, 70],
 1985: [110, 78, 105, 111],
 1986: [85],
 1987: [65],
 1988: [46, 74],
 1989: [125, 76],
 1990: [85, 25, 20, 107],
 1991: [65],
 1992: [18, 106, 108, 105],
 1993: [38, 104, 117, 111],
 1994: [41, 37, 23],
 1995: [18, 37, 11, 23, 107],
 1996: [76, 115, 41],
 1997: [100, 40, 108],
 1998: [85],
 1999: [104, 105, 111],
 2000: [70, 40, 69, 77],
 2001: [66, 56, 109],
 2002: [74, 94, 106],
 2003: [80],
 2004: [97, 77],
 2005: [20, 58, 111],
 2006: [30, 55, 18, 54],
 2007: [105, 108, 44],
 2008: [99, 125, 76, 77],
 2009: [91, 66, 107],
 2010: [90, 19],
 2011: [128, 20, 25],
 2012: [92, 65],
 2013: [54, 55, 111, 33, 34],
 2014: [40, 47, 74, 11, 111],
 2015: [11, 4, 111],
 2016: [41, 77],
 2017: [18, 111],
 2018: [109, 66],
 2019: [60, 105, 111],
 2020: [65],
 2021: [65],
 2022: [48, 27, 8],
 2023: [35, 5, 54, 111],
 2024: [72, 97, 77],
 2025: [18, 108, 111],
 2026: [60, 128, 61],
 2027: [56, 54, 33, 23, 73, 37, 104, 105, 111],
 2028: [92],
 2029: [40, 42, 20, 111],
 2030: [38, 5, 3],
 2031: [74, 98, 66, 105],
 2032: [65],
 2033: [74, 47],
 2034: [110, 11, 21, 104, 111],
 2035: [40, 47, 11, 111],
 2036: [82, 100, 11, 61, 26],
 2037: [54, 55, 37, 18, 23, 47, 25, 20, 66, 105, 107],
 2038: [76, 99, 125],
 2039: [92],
 2040: [46, 47, 101],
 2041: [20, 111],
 2042: [2, 69, 105, 104, 108, 107],
 2043: [86, 121, 70, 99, 84, 125, 126, 118],
 2044: [65],
 2045: [65],
 2046: [44, 78, 97, 77],
 2047: [65],
 2048: [90, 20, 111],
 2049: [18, 108, 111],
 2050: [76, 125],
 2051: [92],
 2052: [104, 105, 111],
 2053: [116, 41, 76, 125, 112],
 2054: [40, 47, 61, 111],
 2055: [18, 104, 111],
 2056: [113],
 2057: [117, 105],
 2058: [111, 11, 21],
 2059: [56, 44, 55, 66],
 2060: [47, 46],
 2061: [74, 94, 47, 23],
 2062: [58, 20, 18, 37, 111],
 2063: [38, 110, 69],
 2064: [74, 47, 111],
 2065: [18, 20, 11, 4],
 2066: [74, 73, 26],
 2067: [58, 56, 79, 33],
 2068: [58, 55, 22, 31, 29, 111],
 2069: [115, 47],
 2070: [11, 20, 47, 111],
 2071: [48, 47, 11, 20, 111],
 2072: [128],
 2073: [110, 25, 20, 111],
 2074: [48, 27, 124],
 2075: [115, 20, 111],
 2076: [100, 36],
 2077: [82, 81],
 2078: [52, 11, 78, 91, 112],
 2079: [113, 11, 40, 47, 107],
 2080: [99, 125, 76, 77],
 2081: [72],
 2082: [99, 97, 77],
 2083: [76, 125],
 2084: [65],
 2085: [65],
 2086: [101],
 2087: [65],
 2088: [65],
 2089: [77, 76],
 2090: [91, 48, 123, 98],
 2091: [46],
 2092: [85, 3, 23],
 2093: [66],
 2094: [85],
 2095: [20, 21, 110, 67],
 2096: [117, 105, 48, 44],
 2097: [20, 110, 80],
 2098: [20, 111, 78],
 2099: [44, 105],
 2100: [106, 108, 104, 105],
 2101: [60, 104, 105, 111],
 2102: [40, 73, 20, 111],
 2103: [35, 111],
 2104: [115, 23, 111],
 2105: [57],
 2106: [47],
 2107: [74, 46],
 2108: [40, 79, 58, 91],
 2109: [40, 73],
 2110: [58, 79, 56, 107],
 2111: [58, 47, 11, 54, 20, 111],
 2112: [73, 37, 111],
 2113: [56, 73, 58],
 2114: [18, 30, 54, 55],
 2115: [27],
 2116: [47, 104, 20, 111],
 2117: [102, 26],
 2118: [42],
 2119: [75, 82],
 2120: [47, 74],
 2121: [119, 120, 69, 37, 74, 100, 23, 80, 97],
 2122: [54, 11, 18, 33, 55],
 2123: [118, 126, 76],
 2124: [86, 87],
 2125: [65],
 2126: [65],
 2127: [65],
 2128: [62],
 2129: [65],
 2130: [65],
 2131: [65],
 2132: [65],
 2133: [101, 114, 69],
 2134: [101, 92],
 2135: [78, 91, 65],
 2136: [78, 98, 108, 44, 105],
 2137: [66, 56, 109],
 2138: [108, 105],
 2139: [18, 44, 104],
 2140: [18, 117, 111],
 2141: [35, 98, 107],
 2142: [37, 78, 110, 111],
 2143: [37, 104, 111],
 2144: [58, 20, 111],
 2145: [58, 20, 33, 26, 111],
 2146: [47, 112],
 2147: [25, 74, 110],
 2148: [74, 39, 73],
 2149: [39, 60, 107],
 2150: [39, 40, 107],
 2151: [74, 47, 111],
 2152: [113, 40, 47, 11, 111],
 2153: [18, 54, 30, 55],
 2154: [54, 37, 11, 73, 55],
 2155: [58, 55, 111],
 2156: [58, 59, 47, 37, 18, 54, 111],
 2157: [115, 111],
 2158: [124, 11, 112],
 2159: [74, 20, 111],
 2160: [26, 111],
 2161: [36, 100],
 2162: [42, 111],
 2163: [86, 111],
 2164: [97, 77, 99, 126],
 2165: [76, 125, 41],
 2166: [127, 114, 68],
 2167: [65],
 2168: [20, 110, 111],
 2169: [110, 78, 111],
 2170: [117],
 2171: [40, 11, 78, 111],
 2172: [11, 40, 18, 23],
 2173: [74, 47, 23, 111],
 2174: [52, 53, 11, 18, 23],
 2175: [40, 11, 111],
 2176: [73, 40, 11, 18, 20],
 2177: [82, 81],
 2178: [99, 76, 125],
 2179: [64, 77],
 2180: [116, 86, 77, 54],
 2181: [65],
 2182: [99, 126, 118, 70],
 2183: [93, 77, 101, 65],
 2184: [65],
 2185: [75],
 2186: [100],
 2187: [44],
 2188: [48, 91, 78],
 2189: [109],
 2190: [85],
 2191: [78, 108, 107],
 2192: [20, 111],
 2193: [92, 82, 81, 65],
 2194: [117, 44, 105],
 2195: [78, 7, 9, 105],
 2196: [104, 108, 105],
 2197: [20, 111, 80],
 2198: [11, 20, 111],
 2199: [90, 20, 78, 111],
 2200: [110, 18, 35, 111],
 2201: [100, 26],
 2202: [44, 78, 57],
 2203: [110, 37, 21, 111],
 2204: [69, 38, 11, 37, 5],
 2205: [100, 74, 104, 105],
 2206: [47, 123, 111],
 2207: [74, 47, 111],
 2208: [115],
 2209: [52, 11, 111],
 2210: [53, 20, 54, 111],
 2211: [61, 66, 91],
 2212: [27],
 2213: [60, 61, 128],
 2214: [36, 26, 102],
 2215: [40, 47, 113, 111],
 2216: [42],
 2217: [82, 24],
 2218: [119, 74, 47],
 2219: [18, 11, 58, 54, 33, 111],
 2220: [52, 40, 69, 11],
 2221: [50, 119, 97, 83],
 2222: [70, 40, 71, 77],
 2223: [76, 116, 41],
 2224: [113, 11, 71, 126, 77],
 2225: [92],
 2226: [65],
 2227: [80, 66],
 2228: [65],
 2229: [65],
 2230: [100, 76, 77, 93],
 2231: [90],
 2232: [85],
 2233: [109],
 2234: [108, 105, 111],
 2235: [26],
 2236: [128, 19, 90, 107],
 2237: [25, 110],
 2238: [104, 18, 111],
 2239: [48, 46, 124],
 2240: [58, 110, 25, 20, 54, 111],
 2241: [25, 110, 91],
 2242: [18, 117, 105],
 2243: [11, 69],
 2244: [57, 109],
 2245: [47, 48, 111, 80, 78],
 2246: [47, 98],
 2247: [47, 40],
 2248: [52, 58, 110, 20, 22, 111],
 2249: [54, 30, 111],
 2250: [52, 53],
 2251: [56, 109, 33, 111],
 2252: [8],
 2253: [54, 33, 20, 111],
 2254: [52, 58, 79],
 2255: [47, 40, 74, 111],
 2256: [100, 26, 8],
 2257: [60, 61, 105],
 2258: [102],
 2259: [42],
 2260: [82, 13, 100, 11, 26],
 2261: [52, 0, 46, 79],
 2262: [41, 47, 18, 69, 112],
 2263: [54, 37, 73, 47, 55, 108, 18, 23],
 2264: [86],
 2265: [125, 99, 119],
 2266: [76, 125, 41],
 2267: [92],
 2268: [101, 114, 127, 69],
 2269: [92, 65],
 2270: [65],
 2271: [65],
 2272: [65],
 2273: [80, 66],
 2274: [65],
 2275: [92, 65],
 2276: [66, 56, 109],
 2277: [91, 66],
 2278: [108, 107],
 2279: [58, 25, 110, 111],
 2280: [18, 105, 44, 78],
 2281: [104, 20, 111],
 2282: [35, 69, 111],
 2283: [35, 69, 11, 4, 6, 122, 111],
 2284: [58, 11, 18, 54, 111],
 2285: [11, 69, 105, 108, 107],
 2286: [56, 11, 21, 111],
 2287: [25, 40, 73],
 2288: [58, 20, 18, 37, 111],
 2289: [38, 110, 57],
 2290: [18, 117, 111],
 2291: [74, 52, 58],
 2292: [58, 59, 47, 38, 54, 22, 111],
 2293: [25, 40, 104],
 2294: [58, 110, 17, 111],
 2295: [115, 47],
 2296: [47, 18, 78, 111],
 2297: [47, 124, 11, 111],
 2298: [60, 105, 20, 78],
 2299: [100, 108, 107],
 2300: [100, 73, 104, 20, 111],
 2301: [100, 104, 111],
 2302: [42, 111],
 2303: [82, 81],
 2304: [58, 11, 54, 20, 25],
 2305: [46, 47, 40, 110, 91, 18, 37, 23, 111, 107],
 2306: [69, 11, 78, 91, 23],
 2307: [97, 77],
 2308: [76, 41, 77],
 2309: [70, 40, 71, 77],
 2310: [68, 97, 77],
 2311: [65],
 2312: [77],
 2313: [62],
 2314: [65],
 2315: [65],
 2316: [76, 77],
 2317: [19],
 2318: [65],
 2319: [108, 105],
 2320: [106, 108, 95],
 2321: [78, 98, 105],
 2322: [108, 107],
 2323: [117, 78],
 2324: [52, 53],
 2325: [18, 111, 106, 117],
 2326: [35, 111],
 2327: [74, 40, 111],
 2328: [11, 69],
 2329: [104, 18, 117, 111],
 2330: [110, 35, 38, 111],
 2331: [46, 23],
 2332: [110, 25, 20, 58, 111],
 2333: [47, 40, 73],
 2334: [74, 48],
 2335: [53, 69, 78, 111],
 2336: [52, 79],
 2337: [40, 61, 107],
 2338: [55, 37, 89, 23],
 2339: [47, 113, 55, 111],
 2340: [40, 74, 79, 65],
 2341: [124, 27],
 2342: [128],
 2343: [60, 61, 20, 105, 98, 111],
 2344: [74, 100],
 2345: [42],
 2346: [82],
 2347: [100, 40, 18, 11, 23],
 2348: [18, 69, 119, 20, 73],
 2349: [40, 18, 11, 47, 74],
 2350: [97, 83, 77, 96],
 2351: [46, 47, 19],
 2352: [81, 82],
 2353: [101],
 2354: [18, 111],
 2355: [92, 97, 93],
 2356: [86, 119, 87],
 2357: [18, 106, 111],
 2358: [52, 79, 107],
 2359: [101, 97, 106, 93],
 2360: [93],
 2361: [42, 111],
 2362: [58, 56, 79, 108],
 2363: [113, 83],
 2364: [65],
 2365: [110, 25, 57, 109],
 2366: [83, 113, 97],
 2367: [117, 105],
 2368: [76, 125, 88, 77],
 2369: [35, 18, 51, 114],
 2370: [38, 44, 105],
 2371: [52, 53, 47, 20, 111],
 2372: [76, 41],
 2373: [76, 125, 97],
 2374: [35, 111],
 2375: [40, 47, 11, 111],
 2376: [70, 76, 126],
 2377: [58, 20, 33, 111],
 2378: [56, 58, 79, 109],
 2379: [100, 73, 104, 105],
 2380: [110, 106, 25],
 2381: [54, 33, 111, 20, 47, 23],
 2382: [70, 114, 83],
 2383: [101],
 2384: [19],
 2385: [82, 81],
 2386: [80],
 2387: [97, 83, 77],
 2388: [38, 44, 110, 107],
 2389: [104, 38, 111],
 2390: [100, 117, 18, 11, 23],
 2391: [18, 11, 47, 23],
 2392: [18, 74, 47, 48, 89, 78],
 2393: [82, 15, 20, 111],
 2394: [108, 78, 98],
 2395: [110, 78, 111],
 2396: [18, 106, 105],
 2397: [25, 110, 74],
 2398: [65],
 2399: [119, 120],
 2400: [57, 109, 56, 89],
 2401: [83],
 2402: [65],
 2403: [70, 126, 76],
 2404: [65],
 2405: [64],
 2406: [126, 119, 125, 99],
 2407: [18, 108, 111],
 2408: [114],
 2409: [57, 56, 44, 33, 26],
 2410: [35, 11, 40, 111],
 2411: [11, 21, 111],
 2412: [82, 20, 15, 111],
 2413: [76, 41, 125, 63],
 2414: [65],
 2415: [62, 101],
 2416: [104, 108, 78],
 2417: [65],
 2418: [62],
 2419: [65],
 2420: [65],
 2421: [70, 99, 126, 118],
 2422: [11, 78, 111],
 2423: [65],
 2424: [86, 121, 119],
 2425: [92],
 2426: [42, 111],
 2427: [58, 47, 20, 111],
 2428: [38, 104, 111],
 2429: [60, 105, 111],
 2430: [66, 91],
 2431: [57, 109],
 2432: [86, 87],
 2433: [117, 11, 23],
 2434: [48, 100, 11, 79, 47, 20, 78, 23],
 2435: [18, 51, 110, 46, 23],
 2436: [59, 47, 104, 111],
 2437: [92, 100],
 2438: [83, 77],
 2439: [76, 77],
 2440: [38, 69, 5],
 2441: [101],
 2442: [48, 85, 107],
 2443: [65],
 2444: [93, 77, 65],
 2445: [107, 105, 108, 57],
 2446: [65],
 2447: [65],
 2448: [65],
 2449: [74, 78, 105],
 2450: [47, 74, 111],
 2451: [52, 53, 47, 20, 111],
 2452: [18, 105, 107],
 2453: [65],
 2454: [84, 41],
 2455: [42, 105, 111],
 2456: [115, 47],
 2457: [86, 118, 126],
 2458: [69, 37, 104, 111],
 2459: [35, 18, 117, 106],
 2460: [83],
 2461: [82, 111],
 2462: [121, 43],
 2463: [77, 76],
 2464: [86, 116, 76, 77, 125],
 2465: [93],
 2466: [5, 104, 111],
 2467: [76, 125],
 2468: [92, 97, 93],
 2469: [65],
 2470: [65],
 2471: [85],
 2472: [97, 83, 77],
 2473: [76],
 2474: [118, 119],
 2475: [86, 87],
 2476: [110, 69, 18, 47, 74],
 2477: [73, 119, 120, 94, 105, 23],
 2478: [52, 53, 11, 18, 47, 54, 58, 59, 110, 20, 15, 82, 61, 122],
 2479: [65],
 2480: [100],
 2481: [60, 105, 20, 111],
 2482: [76, 88],
 2483: [99, 125, 76],
 2484: [86, 119, 94],
 2485: [74, 47, 23, 111, 107],
 2486: [20, 47, 111],
 2487: [44],
 2488: [99, 125, 119, 126, 118],
 2489: [78, 105],
 2490: [88],
 2491: [101],
 2492: [74, 47, 111],
 2493: [44, 117, 105],
 2494: [65],
 2495: [26],
 2496: [76],
 2497: [48, 47, 104, 111],
 2498: [65],
 2499: [90, 48, 110, 78, 111],
 2500: [119, 74],
 2501: [86, 76],
 2502: [88],
 2503: [47, 73, 54, 111],
 2504: [108, 44, 98, 107],
 2505: [123, 103, 124, 27],
 2506: [60, 61, 117, 44],
 2507: [100, 74, 47, 23],
 2508: [92],
 2509: [92],
 2510: [74, 111, 40],
 2511: [58, 56, 79, 109],
 2512: [57, 78, 111],
 2513: [92],
 2514: [25, 74, 110],
 2515: [65],
 2516: [65],
 2517: [65],
 2518: [60, 104, 105, 111],
 2519: [52, 11, 46, 79],
 2520: [40, 11, 47, 20, 25, 78, 69],
 2521: [52, 53, 58, 59, 47, 73, 33, 30, 38, 110, 82],
 2522: [65],
 2523: [65],
 2524: [92],
 2525: [58, 20, 33, 111],
 2526: [65],
 2527: [25, 20, 110],
 2528: [101, 127, 114, 68],
 2529: [110, 52, 25, 20, 111],
 2530: [128],
 2531: [47, 20, 111],
 2532: [110, 82, 67, 17, 12, 16, 15],
 2533: [72, 83],
 2534: [85],
 2535: [48, 85, 26],
 2536: [65],
 2537: [83, 68, 113, 97],
 2538: [50, 64, 83],
 2539: [74, 47, 112, 18],
 2540: [114, 77, 44],
 2541: [114],
 2542: [65],
 2543: [70, 71],
 2544: [8],
 2545: [62],
 2546: [92, 65],
 2547: [79, 52, 107],
 2548: [88],
 2549: [52, 53, 79],
 2550: [107, 108, 106],
 2551: [83, 96],
 2552: [92],
 2553: [92],
 2554: [100],
 2555: [76],
 2556: [65],
 2557: [65],
 2558: [104, 105, 108],
 2559: [82, 38, 117, 10],
 2560: [25, 110, 106],
 2561: [66, 61],
 2562: [52, 11, 18, 23],
 2563: [69, 105, 23, 91, 111],
 2564: [54, 69, 18, 55, 73, 47],
 2565: [11, 104, 111],
 2566: [70, 76, 126],
 2567: [117, 104, 105],
 2568: [92],
 2569: [18, 30, 22, 56, 111],
 2570: [47, 124, 104, 111],
 2571: [65],
 2572: [92],
 2573: [65],
 2574: [88],
 2575: [119, 118],
 2576: [40, 11, 73, 111],
 2577: [65],
 2578: [93, 65],
 2579: [74, 54, 37, 47, 55],
 2580: [88, 76],
 2581: [66, 91, 107],
 2582: [106, 105, 111],
 2583: [114],
 2584: [58, 47, 111],
 2585: [77],
 2586: [70, 86, 99, 119],
 2587: [25, 100, 73],
 2588: [47, 105, 112],
 2589: [47, 100, 46],
 2590: [76, 77],
 2591: [46, 47, 19],
 2592: [101],
 2593: [65],
 2594: [37, 104, 111],
 2595: [126, 118, 99, 125],
 2596: [108, 105, 107],
 2597: [96],
 2598: [92],
 2599: [11, 21, 111],
 2600: [56, 47, 111],
 2601: [48, 123, 27, 46],
 2602: [86],
 2603: [127, 114, 68],
 2604: [115, 112],
 2605: [46, 47, 23, 18, 111],
 2606: [46, 73, 18, 37, 54, 30, 23, 107],
 2607: [54, 33, 11, 18, 23, 55],
 2608: [25, 17, 38, 58, 54],
 2609: [110, 20, 25],
 2610: [100],
 2611: [71, 126, 99],
 2612: [88, 112],
 2613: [38, 104, 111],
 2614: [36, 100],
 2615: [65],
 2616: [85],
 2617: [100, 47, 31, 111],
 2618: [25, 74, 110],
 2619: [47, 74, 111, 23],
 2620: [92],
 2621: [83, 96, 77],
 2622: [118, 126, 76],
 2623: [70, 77, 83],
 2624: [101, 127, 114, 68],
 2625: [92],
 2626: [92],
 2627: [25, 110, 106],
 2628: [65],
 2629: [78, 20, 111],
 2630: [106],
 2631: [92],
 2632: [83, 96, 77],
 2633: [86, 112],
 2634: [65],
 2635: [86, 87],
 2636: [47, 73, 20, 111],
 2637: [74, 47, 111],
 2638: [42],
 2639: [66, 91],
 2640: [100],
 2641: [114, 55, 110, 105],
 2642: [92],
 2643: [76, 99, 125],
 2644: [108, 107],
 2645: [88],
 2646: [100],
 2647: [88, 63],
 2648: [11, 38, 40, 18, 23, 20, 79],
 2649: [18, 11, 47, 74, 25, 20, 107, 112],
 2650: [11, 18, 117, 91, 110, 47, 23],
 2651: [40, 47, 111],
 2652: [83, 126, 41],
 2653: [40, 47, 11, 111],
 2654: [65],
 2655: [115, 23, 111],
 2656: [78, 105, 98],
 2657: [65],
 2658: [110, 25, 80],
 2659: [19],
 2660: [35, 4, 11, 122, 111],
 2661: [85],
 2662: [88, 77],
 2663: [88, 49],
 2664: [126, 83],
 2665: [101],
 2666: [65],
 2667: [117, 111],
 2668: [100, 47, 20, 31, 112],
 2669: [88, 112],
 2670: [65],
 2671: [11, 4, 104, 111],
 2672: [65],
 2673: [47, 40, 113, 111],
 2674: [37, 111],
 2675: [85, 107],
 2676: [88],
 2677: [65],
 2678: [119, 125, 99, 118],
 2679: [118, 99, 76],
 2680: [82, 10],
 2681: [101, 62],
 2682: [48, 47, 11, 20, 111],
 2683: [58, 55, 111],
 2684: [100, 74, 20, 105],
 2685: [54, 55, 38, 11, 107],
 2686: [65],
 2687: [88, 112],
 2688: [100, 74, 11, 20, 111],
 2689: [105, 104, 108],
 2690: [61, 105, 111],
 2691: [18, 117, 105, 23],
 2692: [18, 117, 46, 23, 107],
 2693: [69, 78, 110, 46, 25, 20, 67, 23, 91],
 2694: [65],
 2695: [40, 74],
 2696: [115, 111],
 2697: [65],
 2698: [110, 25, 20, 111],
 2699: [88],
 2700: [93],
 2701: [125, 76, 99],
 2702: [18, 95, 105, 111],
 2703: [86, 121, 119],
 2704: [92, 65],
 2705: [58, 59, 56, 108, 107],
 2706: [48, 85, 123, 103],
 2707: [101, 62],
 2708: [101],
 2709: [92, 65],
 2710: [79, 58, 106, 107],
 2711: [65],
 2712: [65],
 2713: [128],
 2714: [119, 69, 74],
 2715: [42, 111],
 2716: [100, 47, 20, 31, 111],
 2717: [108, 105, 111],
 2718: [38, 104, 111],
 2719: [65],
 2720: [76, 88],
 2721: [93],
 2722: [35, 111, 104],
 2723: [66, 91],
 2724: [36],
 2725: [11, 47, 21, 111],
 2726: [35, 11, 40, 111],
 2727: [110, 20, 57, 78, 111],
 2728: [88],
 2729: [65],
 2730: [18, 30, 22, 56],
 2731: [88, 77],
 2732: [76, 88],
 2733: [86, 76],
 2734: [119, 14, 37],
 2735: [115, 20, 58, 54, 47],
 2736: [11, 40, 47, 110, 117, 18, 69, 78],
 2737: [44, 76, 77],
 2738: [72, 83],
 2739: [125, 88],
 2740: [92],
 2741: [56, 33, 26, 111],
 2742: [47, 123, 104, 111],
 2743: [65],
 2744: [60, 61, 117, 44],
 2745: [47, 23, 74],
 2746: [125, 119],
 2747: [65],
 2748: [25, 110, 78, 111],
 2749: [65],
 2750: [65],
 2751: [105, 104, 108],
 2752: [101],
 2753: [65],
 2754: [114, 44, 105],
 2755: [36],
 2756: [47, 23, 105, 111],
 2757: [88, 49],
 2758: [37, 104, 111],
 2759: [65],
 2760: [42],
 2761: [83, 72, 64],
 2762: [74, 73, 78, 26],
 2763: [65],
 2764: [110, 25, 20, 111],
 2765: [115, 20, 23, 111],
 2766: [65],
 2767: [100],
 2768: [62],
 2769: [69, 104, 111],
 2770: [124, 47, 111],
 2771: [60, 61, 128],
 2772: [88, 43, 49],
 2773: [18, 108, 111],
 2774: [92],
 2775: [101],
 2776: [18, 104, 117, 111],
 2777: [11, 69, 105],
 2778: [11, 91, 78, 23, 111],
 2779: [52, 58, 47, 74, 69, 18, 54, 33, 30, 22]}

knowledge_points = ['For循环',
 'IF条件控制',
 'While循环与Do While循环',
 'break语句',
 'continue语句',
 'do while循环',
 'exit函数',
 'exp',
 'extern关键字',
 'fabs',
 'feof函数',
 'for循环',
 'fputs',
 'fread',
 'free函数',
 'fscanf',
 'fwrite',
 'gets',
 'if语句',
 'main函数',
 'printf',
 'putchar',
 'puts',
 'return语句',
 'rewind',
 'scanf',
 'sizeof',
 'static关键字',
 'static变量',
 'strcat',
 'strcmp',
 'strcpy',
 'strlen',
 'strlen函数',
 'strstr函数',
 'switch语句',
 'typedef',
 'while循环与Do While循环',
 'while循环与do while循环',
 '一维数组介绍',
 '一维数组使用',
 '二分法查找',
 '位运算',
 '先到先服务的作业调度',
 '关系运算',
 '其他运算相关问题',
 '函数与函数原型',
 '函数的调用',
 '变量',
 '后序遍历',
 '堆排序',
 '多分支控制',
 '多维数组',
 '多维数组使用举例',
 '字符串',
 '字符串处理函数',
 '字符串常量',
 '字符常量',
 '字符数组',
 '字符数组使用举例',
 '宏',
 '宏定义',
 '封装',
 '层次遍历',
 '希尔排序',
 '常识',
 '常量',
 '库函数与函数定义',
 '循环结构',
 '循环语句',
 '循环队列',
 '循环队列是队列的一种顺序存储结构',
 '快速排序',
 '指针变量使用举例',
 '指针变量的赋值与引用',
 '数据文件处理',
 '数据结构',
 '数据结构与算法',
 '数据运算与表达式',
 '数组初始化',
 '整型、浮点型与字符串',
 '文件指针是指针类型的变量',
 '文件操作',
 '时间复杂度分析',
 '有序线性表既可以采用顺序存储结构，也可以采用链式存储结构',
 '标识符',
 '栈',
 '栈与指针',
 '树',
 '比较运算',
 '注释',
 '浮点型、整型与字符串',
 '程序和文档',
 '程序流程图',
 '空指针',
 '空语句',
 '空间复杂度分析',
 '算法与程序正确性',
 '类型转换',
 '线性结构',
 '结构体',
 '结构化程序设计',
 '联合',
 '自动变量',
 '自增运算符',
 '表达式求值',
 '语句',
 '语法错误',
 '赋值表达式',
 '转义字符',
 '输入输出基础',
 '输出结果分析',
 '返回结果分析',
 '选择排序',
 '选择结构',
 '递归函数',
 '递归算法与非递归算法',
 '逻辑运算',
 '链式存储结构',
 '链表',
 '链表示例',
 '队列',
 '随机数生成',
 '静态变量',
 '静态局部变量',
 '非线性结构',
 '顺序存储结构',
 '顺序结构',
 '预处理命令']

In [ ]:
# # 数据集演示
# # 多个学生，每个学生都有自己的题目序列和作答结果
# all_student_questions = [
#     [0, 2, 1, 4],        # 学生1
#     [1, 3, 0, 4],        # 学生2
#     [2, 0, 4, 1],        # 学生3
#     # ...
# ]

# all_student_answers = [
#     [1, 0, 1, 1],        # 学生1
#     [0, 1, 1, 0],        # 学生2
#     [1, 1, 0, 1],        # 学生3
#     # ...
# ]

# Q_matrix = {
#     0: [0, 1],    # 题目0涉及知识点0和1
#     1: [2],       # 题目1涉及知识点2
#     2: [1, 3],
#     3: [0],
#     4: [4],
# }

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 假设已知知识点总数和嵌入维度
num_kc = len(knowledge_points)        # 知识点总数（示例）
embed_dim = 32    # 嵌入向量维度

# 构造嵌入层：2*num_kc 个可能的输入类别 (每个知识点的对/错)
kc_embedding = torch.nn.Embedding(num_embeddings=2 * num_kc, embedding_dim=embed_dim)
kc_embedding.weight.requires_grad_(False)  # 仅用于离线编码，禁止跟踪梯度

def encode_interaction(kc_list, is_correct):
    """
    将涉及多个知识点的单条答题记录编码为模型输入向量。
    """
    offset = 0 if is_correct else num_kc
    kc_indices = [kc + offset for kc in kc_list]
    kc_indices = torch.tensor(kc_indices, dtype=torch.long)
    with torch.no_grad():  # 确保输出是常量，避免后续训练复用同一计算图
        vec = kc_embedding(kc_indices).sum(dim=0)
    return vec


def build_dataset(all_student_questions, all_student_answers, Q_matrix, num_kc, embed_dim):
    """
    根据所有学生的题目序列 + 作答结果，构造 DKT 训练用的 X, Y 张量。
    X: (num_students, seq_len-1, embed_dim)
    Y: (num_students, seq_len-1, num_kc)，未涉及的知识点为 -1
    """
    X_list = []
    Y_list = []

    num_students = len(all_student_questions)
    for s in range(num_students):
        questions = all_student_questions[s]
        answers   = all_student_answers[s]

        # 序列长度 L，实际可训练步数是 L-1（因为要预测下一题）
        seq_len = len(questions) - 1

        X_seq = torch.zeros(seq_len, embed_dim)
        Y_seq = torch.full((seq_len, num_kc), -1.0)  # 先全设为 -1，用作掩码

        for t in range(seq_len):
            # 当前步，用来做输入 X[t]
            cur_qid = questions[t]
            cur_ans = answers[t]
            cur_kcs = Q_matrix[cur_qid]
            X_seq[t] = encode_interaction(cur_kcs, cur_ans)

            # 下一步(t+1)，用来做目标 Y[t]
            next_qid = questions[t + 1]
            next_ans = answers[t + 1]
            next_kcs = Q_matrix[next_qid]
            for kc in next_kcs:
                Y_seq[t, kc] = float(next_ans)   # 只填涉及的知识点，其它保持 -1

        X_list.append(X_seq)
        Y_list.append(Y_seq)

    # 堆叠成 (num_students, seq_len, ...) 形式
    X = torch.stack(X_list, dim=0)   # (N, T, embed_dim)
    Y = torch.stack(Y_list, dim=0)   # (N, T, num_kc)
    return X, Y

# ====== 构造整个数据集张量 ======
X_all, Y_all = build_dataset(all_student_questions, all_student_answers, Q_matrix, num_kc, embed_dim)

print("X_all shape:", X_all.shape)  # 预期: (num_students, seq_len-1, embed_dim)
print("Y_all shape:", Y_all.shape)  # 预期: (num_students, seq_len-1, num_kc)

# ====== 打包成 TensorDataset 和 DataLoader ======
dataset = TensorDataset(X_all, Y_all)
data_loader = DataLoader(dataset, batch_size=config.batch_size, shuffle=True)





X_all shape: torch.Size([3, 3, 16])
Y_all shape: torch.Size([3, 3, 5])


## 模型结构定义（LSTM/GRU）

有了输入表示，我们定义DKT模型的网络结构。典型的DKT使用一个单层或多层的LSTM（或GRU）来捕捉时序依赖，其隐藏状态被认为近似表示学生的知识状态。隐藏状态再通过一个全连接输出层映射到每个知识点的预测。

模型结构要点：

嵌入层：将输入的one-hot索引转换为低维向量表示（如前述的kc_embedding）。如果已经手动编码了向量，可以省略这一层。

循环层（LSTM/GRU）：选择LSTM或GRU单元，隐状态维度可调。我们在每个时间步输入当次交互的嵌入向量，隐状态递归更新。

全连接输出层：将LSTM的隐状态映射到大小为num_kc的输出向量，每个维对应一个知识点。我们对输出使用Sigmoid激活，将其约束为0-1概率。

In [ ]:


class DKTModel(nn.Module):
    def __init__(self, num_kc, embed_dim, hidden_dim):
        super(DKTModel, self).__init__()
        self.num_kc = num_kc
        self.hidden_dim = hidden_dim
        # 嵌入层：将2*num_kc维的稀疏输入映射为embed_dim维（如果预先用了encode_interaction手动处理，这里可选）
        self.embedding = nn.Embedding(2 * num_kc, embed_dim)
        # LSTM层：输入维度为embed_dim，隐藏状态维度为hidden_dim，单层LSTM
        self.lstm = nn.LSTM(input_size=embed_dim, hidden_size=hidden_dim, batch_first=True)
        # 输出层：全连接，将hidden_dim映射到每个知识点的掌握概率
        self.output = nn.Linear(hidden_dim, num_kc)
        # 使用Sigmoid将输出转换为概率 [0,1]
        # （训练时我们也可以直接使用nn.BCEWithLogitsLoss而不在这里激活）
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, input_sequences):
        """
        input_sequences:  或 (batch_size, seq_len, embed_dim)
         - 若已提供实数张量 (batch, seq_len, embed_dim)，则直接送入LSTM。
        返回: 输出概率张量 (batch_size, seq_len, num_kc)
        """
        
        # 已经是嵌入后的向量
        embeds = input_sequences
        # 将嵌入序列送入LSTM，输出整个序列的隐状态
        lstm_out, _ = self.lstm(embeds)  # lstm_out: (batch, seq_len, hidden_dim)
        # 通过全连接输出层将每个时间步的隐状态转换为知识点概率
        pred_logits = self.output(lstm_out)         # (batch, seq_len, num_kc)
        pred_probs = self.sigmoid(pred_logits)      # 应用Sigmoid到0-1范围
        return pred_probs


## 损失函数设计（交叉熵）与优化

DKT模型训练通常采用交叉熵损失来比较预测概率与实际结果。由于这里每个知识点输出独立为二分类，我们采用二元交叉熵损失（Binary Cross Entropy）对每个知识点的输出进行评估。具体做法：

掩码损失计算：对每个时间步，我们只关心该题涉及的知识点的输出。因此我们可以构造同维度的目标矩阵Y，在涉及的知识点位置填入实际结果(1或0)，在未涉及位置填入缺失标记（如-1）或0并在损失计算时忽略。例如上节代码中Y_seq矩阵，有值的位置表示那些知识点在该题中的表现。计算损失时，我们针对每个时间步、每个知识点计算BCE(pred, target)，然后用掩码只累计那些target非缺失的项。

实现方法：一种简单实现是逐步手工计算：对于每条序列的每个时间t，取出题目的知识点集合C，对于其中每个知识点$k \in C$，如果实际答对则希望预测$p_k$接近1，答错则$p_k$接近0。可以将损失累加为：$-\sum_{k \in C} [a_t \log p_{t,k} + (1-a_t)\log(1-p_{t,k})]$，然后对整个batch取平均。
这种按知识点计算相当于应用了输出掩码，只更新相关知识点的参数，有助于模型准确跟踪各知识点状态。

In [ ]:
import torch.nn.functional as F

# 模型实例化
model = DKTModel(num_kc=num_kc, embed_dim=embed_dim, hidden_dim=64).to(device)
criterion = nn.BCELoss()         # 二元交叉熵损失 (会对每个输出独立计算然后取平均)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

# 假设我们有批量的学生序列数据 inputs, targets
# inputs: 张量 (batch_size, seq_len, embed_dim) 或 (batch, seq_len) 索引
# targets: 张量 (batch_size, seq_len, num_kc), 未涉及知识点处可以是0但我们需要掩码来忽略
# 为简化演示，假设已经准备好 inputs 和 targets

num_epochs = config.num_epochs
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for inputs, targets in data_loader:  # inputs: (batch, T, embed_dim), targets: (batch, T, num_kc)
        optimizer.zero_grad()

        inputs = inputs.to(device)
        targets = targets.to(device)

        pred = model(inputs)  # (batch, T, num_kc), 已经是 Sigmoid 后的概率

        # --------- 构造掩码 & 目标 ---------
        # mask: 只关心 targets >= 0 的位置（即你填了 0 或 1 的知识点）
        mask = (targets >= 0)                      # (batch, T, num_kc)  bool
        # 把 -1 的地方先变成 0，方便喂进 BCE，反正 mask 会屏蔽掉它们
        target_clamped = targets.clone()
        target_clamped[targets < 0] = 0.0          # (batch, T, num_kc)，只在 mask 的位置有意义

        # --------- 计算 masked BCE loss（一次性算完） ---------
        # 先按元素算 BCE，reduction='none' 保留每个位置
        bce = F.binary_cross_entropy(pred, target_clamped, reduction='none')  # (batch, T, num_kc)

        # 掩码无关位置
        bce = bce * mask.float()                  # 不关心的地方 loss=0

        # 有效位置数量
        valid_count = mask.sum()
        if valid_count == 0:
            continue  # 这批刚好全是 -1，就跳过

        loss = bce.sum() / valid_count

        # --------- 反向传播 & 更新 ---------
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs}, Loss = {total_loss:.4f}")

torch.save(model.state_dict(), "dkt_model.pt")

                


Epoch 1/10, Loss = 0.6893
Epoch 2/10, Loss = 0.6815
Epoch 3/10, Loss = 0.6739
Epoch 4/10, Loss = 0.6663
Epoch 5/10, Loss = 0.6588
Epoch 6/10, Loss = 0.6513
Epoch 7/10, Loss = 0.6438
Epoch 8/10, Loss = 0.6363
Epoch 9/10, Loss = 0.6288
Epoch 10/10, Loss = 0.6213


## DKT模型知识掌握概率预测函数

get_knowledge_probabilities(model, past_interactions) 函数基于一个训练好的深度知识追踪（DKT）模型和某个学生的历史答题记录，计算该学生当前对各知识点的掌握概率。函数会将学生的交互序列输入模型，得到最后一个时间步对应的输出概率向量。该概率向量中的每个元素表示学生对相应知识点的掌握程度（取值范围为0~1），有助于预测学生在未来题目上的表现。

#### 实现思路

序列编码：首先将学生的交互序列转换为模型可接受的输入格式。通常DKT模型将每道题的作答结果编码为一个one-hot向量或embedding索引。例如，每道题目ID qid 对应唯一的知识点，当回答正确时可映射为索引 qid + N，回答错误映射为索引 qid（其中 N 是题目/知识点总数）。这样正确和错误的交互用不同索引加以区分。转换完成后，我们将得到一个由这些索引组成的序列张量。

模型推断：将编码后的序列张量输入DKT模型进行前向传播。模型内部的循环神经网络（如LSTM/GRU）会逐步更新学生的知识状态，并输出各时间步对应的预测。我们在推断时通常只关心最后一个时间步的输出，这代表模型对学生当前知识状态的判断。如果模型的 forward 需要初始隐藏状态，我们会对隐藏状态进行适当初始化（例如全零张量）。在推断过程中使用 torch.no_grad()，确保不记录梯度以提高运行效率。

结果获取：提取模型在最后一个时间步的输出向量，并通过Sigmoid激活函数将其映射到0~1之间，得到每个知识点的掌握概率。最后将这个概率向量返回。函数返回值为一个与知识点数量相同长度的PyTorch张量，其中每个元素即对应知识点的掌握概率。

In [5]:
def predict_correct_prob(model_path, past_interactions, next_question_kc=None, hidden_dim=64):
    """
    给定模型和历史交互，返回对下一题的预测概率（支持输入下题知识点集合）。
    
    Args:
        model: 已训练好的 DKT 模型
        past_interactions: List[(q_id, kc_list, is_correct)] 学生历史交互
        next_question_kc: List[int] 下一题所涉及的知识点编号

    Returns:
        若 next_question_kc 提供：返回对该题答对的预测概率（float）
        否则：返回知识点掌握概率向量 (Tensor)
    """
    state_dict = torch.load(model_path, map_location=device)
    model = DKTModel(num_kc=num_kc, embed_dim=embed_dim, hidden_dim=hidden_dim).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    model_device = device

    if len(past_interactions) == 0:
        # 没有历史记录，返回默认预测
        return 0.5 if next_question_kc else torch.full((model.num_kc,), 0.5)

    # 自动获取编码维度
    seq_len = len(past_interactions)
    X = torch.zeros(1, seq_len, embed_dim)

    for t, (q_id, kc_list, is_correct) in enumerate(past_interactions):
        X[0, t] = encode_interaction(kc_list, is_correct)

    X = X.to(model_device)

    with torch.no_grad():
        pred_seq = model(X)
        last_probs = pred_seq[0, -1].detach().cpu()  # (num_kc,)

    if next_question_kc:
        return last_probs[next_question_kc].mean().item()
    else:
        return last_probs


